# Onboard object detection with compact descriptor transmission for bandwidth-constrained AUV monitoring

Experiment notebook. Runs on Kaggle with a T4 GPU.

**Inputs**
1. URPC2020 in YOLO format (folder with `data.yaml`, `train/` and `valid/`).
2. Optional: trained detector `yolov8n_urpc_best.pt`. If it is not attached, Cell 5 trains YOLOv8-n with seed 42.
3. For session C: RUOD (https://github.com/dlut-dimt/RUOD), in a folder whose name contains `ruod`.

**Sessions** (set `SESSION` in Cell 3)
- `A`: channel experiments, baselines, energy, coverage and link emulation
- `B`: detector training seeds 43 and 44
- `C`: RUOD evaluation and quantization ablations
- `Q`: quantization, confidence threshold and JPEG quality ablations only
- `D`: JSCC at 640 x 640 (optional)

Results are written to `/kaggle/working/results` and packed by Cell 30.


In [ ]:
# Cell 1: Install libraries
import subprocess, sys
for pkg in ['ultralytics==8.4.130', 'torchmetrics', 'pyyaml', 'pycocotools']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
import importlib.metadata as _md
print('Installed ultralytics', _md.version('ultralytics'))


In [ ]:
# Cell 2: Imports, seeding and output folder
import os, glob, json, math, time, shutil, zlib, pickle, random, warnings, platform
import struct, socket, threading, queue
import numpy as np
import pandas as pd
import cv2
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from ultralytics import YOLO

warnings.filterwarnings('ignore')

GLOBAL_SEED = 42


def set_all_seeds(seed):
    # Seeds every random generator used in this notebook
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(GLOBAL_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR = '/kaggle/working/results'
os.makedirs(OUT_DIR, exist_ok=True)
print('Device:', device, '| torch', torch.__version__)


In [ ]:
# Cell 3: Configuration and parameter specification
# Each parameter is registered with spec() together with its unit and source. Cell 28 exports them as a table.

SESSION = 'A'
# A   channel experiments, baselines, energy, coverage and link emulation (about 4 to 5 h on a T4)
# B   detector training seeds 43 and 44 (about 3 h)
# C   RUOD cross dataset evaluation and quantization ablations (about 2 h)
# BC  sessions B and C together
# Q   quantization, confidence threshold and JPEG quality ablations only (about 15 min)
# D   JSCC at 640 x 640 (optional)

SPEC_ROWS = []


def spec(key, value, unit, group, source, kind):
    # Registers a parameter and returns its value
    SPEC_ROWS[:] = [r for r in SPEC_ROWS if r['key'] != key]
    SPEC_ROWS.append(dict(key=key, value=value, unit=unit, group=group, source=source, kind=kind))
    return value


# Sources
REF_MODEM = 'EvoLogics S2C R 18/34 product datasheet (evologics.de/product/s2c-r-18-34-1)'
REF_PI5_BENCH = 'Ultralytics documentation, Raspberry Pi guide: YOLOv8n on Raspberry Pi 5, NCNN, FP32, 640 px'
REF_PI5 = 'Raspberry Pi 5 product brief and official 27 W USB-C power supply (5.1 V, 5 A)'
REF_JETSON = 'NVIDIA Jetson Orin NX series data sheet (power modes)'
REF_STOJ = 'M. Stojanovic, On the relationship between capacity and distance in an underwater acoustic channel, 2007'
REF_URICK = 'R. J. Urick, Principles of Underwater Sound, 3rd ed.'

G = 'Detector and descriptor'
Q_POS = spec('Q_POS', 1024, 'levels', G, 'Ablation in Cell 17', 'design choice')
Q_CONF = spec('Q_CONF', 255, 'levels', G, '8-bit confidence', 'design choice')
CONF_THR = spec('CONF_THR', 0.10, '-', G, 'Highest mAP in the threshold ablation (Cell 17B)', 'design choice')
NMS_IOU_THR = spec('NMS_IOU_THR', 0.45, '-', G, 'NMS IoU threshold', 'design choice')
IMGSZ = spec('IMGSZ', 640, 'px', G, 'YOLOv8 training and inference resolution', 'design choice')
HEADER_BITS = spec('HEADER_BITS', 16, 'bit', G, 'Detection count per image', 'design choice')
JPEG_Q_MAIN = spec('JPEG_Q_MAIN', 50, '-', G, 'Mid-range JPEG quality; ablation at 25 and 75', 'design choice')

G = 'Acoustic modem'
MODEM_NAME = spec('MODEM_NAME', 'EvoLogics S2C R 18/34', '-', G, REF_MODEM, 'datasheet')
MODEM_F_LOW_KHZ = spec('MODEM_F_LOW_KHZ', 18.0, 'kHz', G, REF_MODEM, 'datasheet')
MODEM_F_HIGH_KHZ = spec('MODEM_F_HIGH_KHZ', 34.0, 'kHz', G, REF_MODEM, 'datasheet')
MODEM_MAX_RATE_BPS = spec('MODEM_MAX_RATE_BPS', 13900, 'bit/s', G, REF_MODEM + ': up to 13.9 kbit/s', 'datasheet')
MODEM_MAX_RANGE_M = spec('MODEM_MAX_RANGE_M', 3500, 'm', G, REF_MODEM, 'datasheet')
MODEM_TX_POWER_TABLE = spec('MODEM_TX_POWER_TABLE', {1000: 2.8, 2000: 8.0, 3500: 35.0}, 'W at range (m)', G,
                            REF_MODEM + ': transmit power per range setting', 'datasheet')
MODEM_TX_POWER_MAX_W = spec('MODEM_TX_POWER_MAX_W', 65.0, 'W', G, REF_MODEM, 'datasheet')
MODEM_RX_POWER_W = spec('MODEM_RX_POWER_W', 0.8, 'W', G, REF_MODEM, 'datasheet')
MODEM_LISTEN_POWER_W = spec('MODEM_LISTEN_POWER_W', 0.285, 'W', G, REF_MODEM + ': upper end of 5 to 285 mW', 'datasheet')
MODEM_DEPTH_RATING_M = spec('MODEM_DEPTH_RATING_M', 2000, 'm', G, REF_MODEM + ': metal housings', 'datasheet')
LINK_RATE_BPS = spec('LINK_RATE_BPS', 10000, 'bit/s', G,
                     '72 % of the 13.9 kbit/s datasheet maximum, margin for coding and multipath', 'design choice')
OPERATING_RANGE_M = spec('OPERATING_RANGE_M', 1000, 'm', G, 'Datasheet range of the lowest power setting', 'design choice')
TX_POWER_W = spec('TX_POWER_W', MODEM_TX_POWER_TABLE[OPERATING_RANGE_M], 'W', G,
                  REF_MODEM + ': power for the operating range', 'datasheet')
FRAME_PAYLOAD_BITS = spec('FRAME_PAYLOAD_BITS', 512, 'bit', G, '64-byte frame payload, implemented in Cell 25', 'design choice')
FRAME_OVERHEAD_BITS = spec('FRAME_OVERHEAD_BITS', 64, 'bit', G, '32-bit frame header + CRC-32, implemented in Cell 25',
                           'design choice')
FRAME_PREAMBLE_S = spec('FRAME_PREAMBLE_S', 0.025, 's', G, 'Not in the datasheet; sensitivity 0 to 0.1 s in Cell 19',
                        'assumption')
MODULATION = spec('MODULATION', 'bpsk', '-', G, 'Modem waveform is proprietary; uncoded BPSK is a conservative reference',
                  'assumption')
FADING = spec('FADING', 'awgn', '-', G, 'Fades are modelled separately by the Gilbert-Elliott burst model', 'assumption')
CODING_GAIN_DB = spec('CODING_GAIN_DB', 0.0, 'dB', G, 'No coding gain credited (conservative)', 'assumption')
JSCC_BITS_PER_SYMBOL = spec('JSCC_BITS_PER_SYMBOL', 4, 'bit/symbol', G, '16-QAM equivalence used only to count JSCC frames',
                            'design choice')

G = 'Propagation and coverage'
SPREADING_K = spec('SPREADING_K', 1.5, '-', G, REF_STOJ + ' (practical spreading)', 'literature')
SHIPPING_S = spec('SHIPPING_S', 0.5, '-', G, REF_STOJ + ' (moderate shipping)', 'literature')
WIND_MPS = spec('WIND_MPS', 5.0, 'm/s', G, 'Moderate sea state; sensitivity 0 to 15 m/s in Cell 24', 'assumption')
SOUND_SPEED_MPS = spec('SOUND_SPEED_MPS', 1500.0, 'm/s', G, REF_URICK, 'literature')
TX_EFFICIENCY = spec('TX_EFFICIENCY', 0.25, '-', G, 'Cancels out after calibration to the datasheet power-range pairs',
                     'assumption')
DI_DB = spec('DI_DB', 0.0, 'dB', G, REF_MODEM + ': horizontally omnidirectional transducer', 'datasheet')
FER_REQ = spec('FER_REQ', 0.01, '-', G, 'Frame error rate that defines the edge of the datasheet range', 'design choice')
MAP_TARGET = spec('MAP_TARGET', 0.60, 'mAP@0.5', G, 'Minimum useful detection quality for coverage', 'design choice')
FPS_TARGETS = spec('FPS_TARGETS', [0.05, 1, 5], 'fps', G,
                   'Monitoring rates for coverage; 0.05 fps lets the image-based baselines appear at all', 'design choice')
TDMA_GUARD_COMP_S = spec('TDMA_GUARD_COMP_S', 0.05, 's', G,
                         'TDMA guard when AUVs pre-compensate their known propagation delay (modem ranging)', 'assumption')
COVER_MAX_RANGE_M = spec('COVER_MAX_RANGE_M', 6000, 'm', G, 'Coverage grid extent', 'design choice')
COVER_MAX_DEPTH_M = spec('COVER_MAX_DEPTH_M', 1000, 'm', G, 'Coverage grid depth, within the 2000 m rating', 'design choice')

G = 'Edge platform'
PI5_LATENCY_MS = spec('PI5_LATENCY_MS', 94.28, 'ms/image', G, REF_PI5_BENCH, 'official benchmark')
PI5_POWER_W = spec('PI5_POWER_W', 8.8, 'W', G,
                   'Typical full-load draw; results checked up to PI5_POWER_MAX_W',
                   'assumption')
PI5_POWER_MAX_W = spec('PI5_POWER_MAX_W', 25.5, 'W', G, REF_PI5 + ': hard upper bound of the official supply', 'datasheet')
PI5_PRICE_USD = spec('PI5_PRICE_USD', 80.0, 'USD', G, 'Raspberry Pi official list price, 8 GB',
                     'datasheet')
JETSON_POWER_MODES_W = spec('JETSON_POWER_MODES_W', (10.0, 25.0), 'W', G, REF_JETSON + ' (Orin NX 16 GB)', 'datasheet')
JETSON_LATENCY_MS = spec('JETSON_LATENCY_MS', None, 'ms/image', G,
                         'Optional; row skipped while None', 'official benchmark')
JETSON_PRICE_USD = spec('JETSON_PRICE_USD', 999.0, 'USD', G, 'NVIDIA list price, Orin NX 16 GB',
                        'datasheet')
CAMERA_NAME = spec('CAMERA_NAME', 'Raspberry Pi Camera Module 3', '-', G, 'Raspberry Pi official product page', 'datasheet')
CAMERA_PRICE_USD = spec('CAMERA_PRICE_USD', 25.0, 'USD', G, 'Raspberry Pi official list price',
                        'datasheet')
INFER_POWER_W = PI5_POWER_W
INFER_TIME_S = PI5_LATENCY_MS / 1000.0

G = 'Mission and cost scenario'
MISSION_HOURS = spec('MISSION_HOURS', 4.0, 'h', G, 'Typical survey mission length', 'design choice')
MISSIONS_PER_YEAR = spec('MISSIONS_PER_YEAR', 150, '-', G, 'Operating scenario', 'design choice')
YEARS = spec('YEARS', 3, 'years', G, 'Total cost of ownership horizon', 'design choice')
PRIMARY_BATTERY_USD_PER_WH = spec('PRIMARY_BATTERY_USD_PER_WH', 1.0, 'USD/Wh', G,
                                  'Single-use lithium pack cost; ratios between schemes do not depend on it', 'assumption')
GRID_USD_PER_KWH = spec('GRID_USD_PER_KWH', 0.20, 'USD/kWh', G, 'Rechargeable pack charged from shore power', 'assumption')

G = 'Evaluation protocol'
LOSS_SEEDS = spec('LOSS_SEEDS', [42, 43, 44, 45, 46], '-', G, 'Five loss seeds', 'design choice')
JSCC_SEEDS = spec('JSCC_SEEDS', [42, 43, 44], '-', G, 'Fewer seeds for the image-based learned baselines', 'design choice')
TRAIN_SEEDS = spec('TRAIN_SEEDS', [43, 44], '-', G, 'Independent detector trainings besides seed 42', 'design choice')
SNR_LIST_LEGACY = spec('SNR_LIST_LEGACY', [10, 20, 30], 'dB', G, 'Legacy grid', 'design choice')
RHO_LIST = spec('RHO_LIST', [0.0, 0.05, 0.10], '-', G, 'Legacy grid', 'design choice')
SNR_LIST_BER = spec('SNR_LIST_BER', [0, 2, 4, 6, 8, 10, 12, 14], 'dB Es/N0', G, 'Covers the full frame error transition',
                    'design choice')
GE_MEAN_LOSS = spec('GE_MEAN_LOSS', 0.10, '-', G, 'Same mean loss as the iid worst case', 'design choice')
GE_BURST_LENGTHS = spec('GE_BURST_LENGTHS', [2, 4, 8], 'frames', G, 'Mean burst lengths of the Gilbert-Elliott model',
                        'design choice')
GE_LOSS_GOOD = 0.0
GE_LOSS_BAD = 1.0

# JSCC baselines
JSCC_C = spec('JSCC_C', 16, 'channels', G, '32 x 32 x 16 latent', 'design choice')
JSCC_IMG_HW = spec('JSCC_IMG_HW', 256, 'px', G, 'JSCC input resolution', 'design choice')
JSCC640_C = spec('JSCC640_C', 3, 'channels', G, '80 x 80 x 3 latent, close to the 256 px channel uses', 'design choice')
DEEPJSCC_C = spec('DEEPJSCC_C', 16, 'channels', G, 'Same channel uses as the JSCC autoencoder', 'design choice')
JSCC_TRAIN_SNRS = sorted(set(SNR_LIST_LEGACY + SNR_LIST_BER))

# Training lengths
YOLO_EPOCHS = spec('YOLO_EPOCHS', 50, 'epochs', G, 'Detector training length', 'design choice')
JSCC_EPOCHS = spec('JSCC_EPOCHS', 50, 'epochs', G, 'JSCC training length', 'design choice')
DEEPJSCC_EPOCHS = spec('DEEPJSCC_EPOCHS', 50, 'epochs', G, 'Same training length as the JSCC autoencoder', 'design choice')
JSCC640_EPOCHS = spec('JSCC640_EPOCHS', 20, 'epochs', G, 'GPU budget', 'design choice')
RUOD_EPOCHS = spec('RUOD_EPOCHS', 30, 'epochs', G, 'GPU budget, generalisation check only', 'design choice')
QUANT_LEVELS = spec('QUANT_LEVELS', [256, 512, 1024, 2048], 'levels', G, 'Quantization ablation in Cell 17',
                    'design choice')

# Section switches, set by SESSION
_S = SESSION.upper()
_A = _S in ('A', 'ALL')
RUN_LEGACY_REPRO = RUN_JSCC = RUN_DEEPJSCC = RUN_BER_SWEEP = RUN_BURST = _A
RUN_FORMAT_ABLATION = RUN_PER_CLASS = RUN_BUDGET = _A
RUN_QUANT_ABLATION = _A or _S in ('C', 'BC', 'Q')
RUN_SPLIT = RUN_COMPLEXITY = RUN_COST = RUN_COVERAGE = RUN_SIL = _A
RUN_RES_CONTROL = _A or _S == 'D'
RUN_JSCC640 = _S in ('D', 'ALL')
RUN_TRAIN_SEEDS = _S in ('B', 'BC', 'ALL')
RUN_RUOD = _S in ('C', 'BC', 'ALL')

# RNG stream tags
TAG_JPEG = 0
TAG_SEM = 1
TAG_ORDER = 99
TAG_JSCC = 7

print(f'Session {_S}. Modem: {MODEM_NAME}, link {LINK_RATE_BPS} bit/s, {len(SPEC_ROWS)} parameters registered.')


In [ ]:
# Cell 4: Dataset discovery
INPUT_DIR = '/kaggle/input'
WORK_DIR = '/kaggle/working'


def find_split(base, names):
    for n in names:
        p = os.path.join(base, n, 'images')
        if os.path.isdir(p) and len(os.listdir(p)) > 0:
            return p
    return None


def find_urpc_yaml():
    # Uses the data.yaml that sits next to non-empty train and valid (or val) image folders
    cands = []
    for root, dirs, files in os.walk(INPUT_DIR):
        if 'ruod' in root.lower():
            continue
        if 'data.yaml' in files:
            ok = find_split(root, ['train']) and find_split(root, ['valid', 'val'])
            cands.append((root, bool(ok)))
    for root, ok in cands:
        print(('  use   ' if ok else '  skip  ') + root)
    good = [r for r, ok in cands if ok]
    if not good:
        raise FileNotFoundError('No URPC folder with data.yaml + train/images + valid/images found under /kaggle/input. '
                                f'data.yaml files seen: {[r for r, _ in cands]}')
    return good[0]


src = find_urpc_yaml()
URPC_DIR = os.path.join(WORK_DIR, 'dataset-taskoriented')
if os.path.exists(URPC_DIR) and not (find_split(URPC_DIR, ['train']) and find_split(URPC_DIR, ['valid', 'val'])):
    shutil.rmtree(URPC_DIR)          # a wrong copy from an earlier attempt
if not os.path.exists(URPC_DIR):
    shutil.copytree(src, URPC_DIR)

TRAIN_IMG = find_split(URPC_DIR, ['train'])
VAL_IMG = find_split(URPC_DIR, ['valid', 'val'])
VAL_LBL = VAL_IMG.replace('images', 'labels')
URPC_YAML = os.path.join(URPC_DIR, 'data.yaml')
with open(URPC_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'], cfg['train'], cfg['val'] = URPC_DIR, TRAIN_IMG, VAL_IMG
with open(URPC_YAML, 'w') as f:
    yaml.safe_dump(cfg, f)

names = cfg['names']
CLASS_NAMES = list(names.values()) if isinstance(names, dict) else list(names)
NC = len(CLASS_NAMES)
VAL_FILES = sorted(glob.glob(os.path.join(VAL_IMG, '*.*')))
print('URPC classes:', CLASS_NAMES, '| validation images:', len(VAL_FILES))


PREV_DETECTOR_PATH = ''   # optional, full path of trained detector weights


_pts = sorted(glob.glob(f'{INPUT_DIR}/**/*.pt', recursive=True))
print('Weight files found in the inputs:', len(_pts))
for p in _pts[:30]:
    print('  ', p, f'({os.path.getsize(p) / 1e6:.1f} MB)')


In [ ]:
# Cell 5: Load or train the detector


def find_in_inputs(filename):
    hits = sorted(glob.glob(f'{INPUT_DIR}/**/{filename}', recursive=True))
    return hits[0] if hits else None


def train_or_load(name, pretrained, data_yaml, seed, epochs=YOLO_EPOCHS, reuse_path=None):
    # Loads existing weights (this session or the inputs), otherwise trains with the given seed.
    dst = f'{WORK_DIR}/runs/{name}/weights/best.pt'
    if os.path.exists(dst):
        return YOLO(dst), dst
    if reuse_path:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(reuse_path, dst)
        print('Reused weights from the input:', reuse_path)
        return YOLO(dst), dst
    print(f'Training {name} with seed {seed} for {epochs} epochs')
    YOLO(pretrained).train(data=data_yaml, epochs=epochs, imgsz=IMGSZ, batch=16,
                           project=f'{WORK_DIR}/runs', name=name, exist_ok=True,
                           verbose=False, seed=seed, deterministic=True)
    return YOLO(dst), dst


prev_det = PREV_DETECTOR_PATH or find_in_inputs('yolov8n_urpc_best.pt')
if prev_det:
    print('Using trained detector from the inputs:', prev_det)
else:
    print('No trained detector in the inputs; training YOLOv8 n with seed', GLOBAL_SEED)
detector, DET_WEIGHTS = train_or_load('train_n', 'yolov8n.pt', URPC_YAML, GLOBAL_SEED, reuse_path=prev_det)

# A JSCC autoencoder trained with this detector run is reused when present in the inputs.
os.makedirs(OUT_DIR, exist_ok=True)
prev_jscc = find_in_inputs('jscc256.pt')
if prev_jscc and not os.path.exists(f'{OUT_DIR}/jscc256.pt'):
    shutil.copy(prev_jscc, f'{OUT_DIR}/jscc256.pt')
    print('Reused JSCC autoencoder from the inputs:', prev_jscc)

print('Detector:', DET_WEIGHTS, '| ultralytics', __import__('ultralytics').__version__)


In [ ]:
# Cell 6: Evaluation helpers


def load_gt(lbl_path, w, h):
    boxes, cls = [], []
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 5:
                    continue
                c, cx, cy, bw, bh = map(float, p[:5])
                boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
                cls.append(int(c))
    return np.array(boxes, np.float32).reshape(-1, 4), np.array(cls, int)


def get_yolo_preds(model, img_bgr, conf_thr=CONF_THR, iou_thr=NMS_IOU_THR, imgsz=IMGSZ):
    r = model(img_bgr, conf=conf_thr, iou=iou_thr, imgsz=imgsz, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return np.zeros((0, 4), np.float32), np.zeros(0, np.float32), np.zeros(0, int)
    return (r.boxes.xyxy.cpu().numpy().astype(np.float32),
            r.boxes.conf.cpu().numpy().astype(np.float32),
            r.boxes.cls.cpu().numpy().astype(int))


def make_pred_dict(b, s, c):
    return dict(boxes=torch.tensor(np.asarray(b, np.float32).reshape(-1, 4)),
                scores=torch.tensor(np.asarray(s, np.float32).reshape(-1)),
                labels=torch.tensor(np.asarray(c, np.int64).reshape(-1)))


def compute_map_full(preds, gts):
    m = MeanAveragePrecision(iou_thresholds=[0.5 + 0.05 * i for i in range(10)])
    m.update(preds, gts)
    r = m.compute()
    return {'map50': float(r['map_50']), 'map50_95': float(r['map'])}


def compute_per_class_ap(preds, gts, class_names):
    m = MeanAveragePrecision(iou_thresholds=[0.5], class_metrics=True)
    m.update(preds, gts)
    r = m.compute()
    aps = r['map_per_class'].tolist() if 'map_per_class' in r else []
    if not isinstance(aps, list):
        aps = [aps]
    return {n: (float(aps[i]) if i < len(aps) else float('nan')) for i, n in enumerate(class_names)}


def box_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + bb[None, :] - inter + 1e-9)


def match_tp(pb, ps, pc, gb, gc, iou_thr=0.5):
    # Greedy per class matching at IoU 0.5, highest score first. Returns a true positive flag per prediction.
    tp = np.zeros(len(pb), bool)
    for cls in np.unique(np.concatenate([pc, gc])) if len(pc) + len(gc) else []:
        pi = np.where(pc == cls)[0]; gi = np.where(gc == cls)[0]
        if len(pi) == 0 or len(gi) == 0:
            continue
        order = pi[np.argsort(-ps[pi], kind='stable')]
        ious = box_iou(pb[order], gb[gi]); used = np.zeros(len(gi), bool)
        for r, p in enumerate(order):
            cand = np.where(used, -1.0, ious[r]); j = int(np.argmax(cand))
            if cand[j] >= iou_thr:
                tp[p] = True; used[j] = True
    return tp


In [ ]:
# Cell 7: Descriptor formats

TEXT_BITS_PER_DET = 8 * 46  # replaced in Cell 9 by the measured mean line length


def coord_bits(q_pos=Q_POS):
    # Bits for one coordinate on a grid of q_pos levels, 10 bits for the default 1024 levels
    return max(1, int(math.ceil(math.log2(q_pos))))


def class_bits(nc):
    return max(1, int(math.ceil(math.log2(max(nc, 2)))))


FORMATS = {
    'text_yolo': lambda nc, q=Q_POS: TEXT_BITS_PER_DET,       # one text line c cx cy w h conf, 6 and 4 decimals
    'float32': lambda nc, q=Q_POS: 6 * 32,                     # raw detector row x1 y1 x2 y2 score class
    'float16': lambda nc, q=Q_POS: 6 * 16,                     # same six fields in half precision
    'fixed_bytealigned': lambda nc, q=Q_POS: 80,               # 4 x uint16, uint8, uint8
    'fixed_bitpacked': lambda nc, q=Q_POS: 4 * coord_bits(q) + class_bits(nc) + 8,  # coords, class, 8 bit score
}


def encode_decode(fmt, b, s, c, w, h, q_pos=Q_POS):
    # Returns the detections exactly as the receiver reconstructs them
    if len(b) == 0:
        return b, s, c
    if fmt == 'float32':
        return b.astype(np.float32), s.astype(np.float32), c
    cx = ((b[:, 0] + b[:, 2]) / 2) / w; cy = ((b[:, 1] + b[:, 3]) / 2) / h
    bw = (b[:, 2] - b[:, 0]) / w; bh = (b[:, 3] - b[:, 1]) / h
    f = np.stack([cx, cy, bw, bh], 1).astype(np.float64)
    if fmt == 'text_yolo':
        f = np.round(f, 6)
        sc = np.round(s.astype(np.float64), 4)
    elif fmt == 'float16':
        f = f.astype(np.float16).astype(np.float64)
        sc = s.astype(np.float16).astype(np.float64)
    elif fmt == 'fixed_bytealigned':
        # Same truncation as the previous code, value range 0 to Q_POS inside a 16 bit slot
        f = np.clip(f * q_pos, 0, q_pos).astype(np.int64) / q_pos
        sc = np.clip(s * Q_CONF, 0, Q_CONF).astype(np.int64) / Q_CONF
    elif fmt == 'fixed_bitpacked':
        # Same truncation, value range 0 to Q_POS minus 1 so each coordinate fits its bit width exactly
        f = np.clip(f * q_pos, 0, q_pos - 1).astype(np.int64) / q_pos
        sc = np.clip(s * Q_CONF, 0, Q_CONF).astype(np.int64) / Q_CONF
    else:
        raise ValueError(fmt)
    cx, cy, bw, bh = f[:, 0], f[:, 1], f[:, 2], f[:, 3]
    out = np.stack([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h], 1)
    return out.astype(np.float32), sc.astype(np.float32), c


def pack_bitpacked(b, s, c, w, h, nc, q_pos=Q_POS):
    # Real bit packing into a byte string. Used to prove that the bit count of fixed_bitpacked is achievable.
    cb = class_bits(nc); kb = coord_bits(q_pos); acc = 0; nbits = 0
    for i in range(len(b)):
        cx = ((b[i, 0] + b[i, 2]) / 2) / w; cy = ((b[i, 1] + b[i, 3]) / 2) / h
        bw = (b[i, 2] - b[i, 0]) / w; bh = (b[i, 3] - b[i, 1]) / h
        vals = [int(np.clip(v * q_pos, 0, q_pos - 1)) for v in (cx, cy, bw, bh)]
        fields = [(v, kb) for v in vals] + [(int(c[i]), cb), (int(np.clip(s[i] * Q_CONF, 0, Q_CONF)), 8)]
        for v, wd in fields:
            acc = (acc << wd) | v; nbits += wd
    pad = (-nbits) % 8
    return (acc << pad).to_bytes((nbits + pad) // 8, 'big') if nbits else b'', nbits


def unpack_bitpacked(data, n_det, nc, w, h, q_pos=Q_POS):
    cb = class_bits(nc); kb = coord_bits(q_pos); per = 4 * kb + cb + 8; total = per * n_det
    acc = int.from_bytes(data, 'big') >> ((len(data) * 8) - total) if n_det else 0
    boxes, scores, cls = [], [], []
    for i in range(n_det):
        rec = (acc >> (total - per * (i + 1))) & ((1 << per) - 1)
        sc = rec & 0xFF; cl = (rec >> 8) & ((1 << cb) - 1); co = rec >> (8 + cb)
        bh, bw, cy, cx = [((co >> (kb * k)) & ((1 << kb) - 1)) / q_pos for k in range(4)]
        boxes.append([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h])
        scores.append(sc / Q_CONF); cls.append(cl)
    return np.array(boxes, np.float32).reshape(-1, 4), np.array(scores, np.float32), np.array(cls, int)


In [ ]:
# Cell 8: Channel and framing model
# Frame errors from Es/N0, iid and Gilbert-Elliott frame loss, selective repeat ARQ.


def per_image_rng(seed, img_idx, tag):
    # Independent random stream per image and per method
    return np.random.RandomState([int(seed), int(img_idx), int(tag)])


def bits_per_symbol(modulation=MODULATION):
    return {'bpsk': 1, 'qpsk': 2, '16qam': 4}[modulation]


def ber_from_snr(es_n0_db, modulation=MODULATION, fading=FADING, coding_gain_db=CODING_GAIN_DB):
    # Uncoded Gray mapped bit error rate. SNR is Es over N0 and is converted to Eb over N0 here.
    eb_n0_db = es_n0_db - 10 * math.log10(bits_per_symbol(modulation)) + coding_gain_db
    g = 10 ** (eb_n0_db / 10)
    if modulation in ('bpsk', 'qpsk'):
        a, k = g, 0.5
    else:
        a, k = 0.4 * g, 0.375
    if fading == 'awgn':
        return min(0.5, k * math.erfc(math.sqrt(a)))
    return min(0.5, k * (1 - math.sqrt(a / (1 + a))))


def frame_error_rate(es_n0_db, frame_bits=FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS):
    return 1 - (1 - ber_from_snr(es_n0_db)) ** frame_bits


def frame_time_s(preamble_s=None):
    p = FRAME_PREAMBLE_S if preamble_s is None else preamble_s
    return p + (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS) / LINK_RATE_BPS


def ge_model(mean_loss, burst_len, snr_db=None):
    p_bg = 1.0 / burst_len
    p_gb = mean_loss * p_bg / (1 - mean_loss)
    m = {'type': 'ge', 'p_gb': p_gb, 'p_bg': p_bg, 'loss_good': GE_LOSS_GOOD, 'loss_bad': GE_LOSS_BAD}
    if snr_db is not None:
        m['snr_db'] = snr_db
    return m


def draw_frame_losses(n, model, rng):
    # Returns a boolean array, True means the frame is lost
    t = model['type']
    if n <= 0:
        return np.zeros(0, bool)
    if t == 'none':
        return np.zeros(n, bool)
    if t == 'stream':
        return model['chan'].draw(n)
    if t == 'iid':
        return rng.rand(n) < model['rho']
    if t == 'ber':
        return rng.rand(n) < frame_error_rate(model['snr_db'])
    if t == 'ge':
        p_good = model['loss_good']
        if 'snr_db' in model:
            p_good = max(p_good, frame_error_rate(model['snr_db']))
        pi_b = model['p_gb'] / (model['p_gb'] + model['p_bg'])
        bad = rng.rand() < pi_b
        lost = np.zeros(n, bool)
        for k in range(n):
            lost[k] = rng.rand() < (model['loss_bad'] if bad else p_good)
            bad = (rng.rand() >= model['p_bg']) if bad else (rng.rand() < model['p_gb'])
        return lost
    raise ValueError(t)


class GEStream:
    # Gilbert Elliott channel whose good or bad state continues across frames and across images
    def __init__(self, model, rng):
        self.m = model; self.rng = rng; self.history = []
        self.bad = rng.rand() < model['p_gb'] / (model['p_gb'] + model['p_bg'])

    def draw(self, n):
        m = self.m; out = np.zeros(n, bool)
        p_good = m['loss_good'] if 'snr_db' not in m else max(m['loss_good'], frame_error_rate(m['snr_db']))
        for k in range(n):
            out[k] = self.rng.rand() < (m['loss_bad'] if self.bad else p_good)
            self.bad = (self.rng.rand() >= m['p_bg']) if self.bad else (self.rng.rand() < m['p_gb'])
        self.history.extend(out.tolist())
        return out

    def longest_loss_run(self):
        return longest_run(np.array(self.history, bool))


def longest_run(flags):
    # Length of the longest run of True values
    best = cur = 0
    for f in np.asarray(flags, bool):
        cur = cur + 1 if f else 0
        best = max(best, cur)
    return int(best)


def arq_delivery_time(n_frames, fer, range_m=None, preamble_s=None):
    # Expected delivery time of n_frames with selective repeat ARQ
    if n_frames <= 0:
        return 0.0
    if fer >= 1:
        return float('inf')
    rng_m = OPERATING_RANGE_M if range_m is None else range_m
    rtt = 2.0 * rng_m / SOUND_SPEED_MPS
    rounds = 1.0 + sum(1.0 - (1.0 - fer ** r) ** n_frames for r in range(1, 400))
    return n_frames / (1.0 - fer) * frame_time_s(preamble_s) + (rounds - 1.0) * rtt


def transmit_descriptors(b, s, c, w, h, fmt, nc, loss_model, rng, order='as_is', rng_order=None,
                         max_frames=None, legacy=False, q_pos=Q_POS):
    # legacy=True sends one descriptor per packet
    db, ds, dc = encode_decode(fmt, b, s, c, w, h, q_pos)
    n = len(db); L = FORMATS[fmt](nc, q_pos)
    idx = np.arange(n)
    if order == 'conf_desc':
        idx = np.argsort(-s, kind='stable')
    elif order == 'random':
        idx = rng_order.permutation(n)
    per = 1 if legacy else max(1, FRAME_PAYLOAD_BITS // L)
    nf = max(1, int(math.ceil(n / per)))
    nf_sent = nf if max_frames is None else max(0, min(nf, int(max_frames)))
    lost = draw_frame_losses(nf_sent, loss_model, rng)
    frame_of = np.arange(n) // per
    sent = frame_of < nf_sent
    recv = sent.copy()
    if nf_sent > 0:
        recv[sent] = ~lost[frame_of[sent]]
    sel = idx[recv]
    n_sent = int(sent.sum())
    stats = dict(info_bits=HEADER_BITS + L * n_sent,
                 chan_bits=nf_sent * (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS),
                 n_frames=nf_sent, airtime=nf_sent * frame_time_s(),
                 n_det=n, n_sent=n_sent, n_recv=int(recv.sum()))
    return db[sel], ds[sel], dc[sel], stats


def transmit_image(n_bits, loss_model, rng, legacy=False):
    # Image based methods. Baseline JPEG cannot decode after a lost frame, so any loss drops the image.
    if legacy:
        lost = rng.rand() < loss_model['rho']
        nf = int(math.ceil(n_bits / FRAME_PAYLOAD_BITS))
    else:
        nf = int(math.ceil(n_bits / FRAME_PAYLOAD_BITS))
        lost = bool(draw_frame_losses(nf, loss_model, rng).any())
    return lost, dict(info_bits=n_bits, chan_bits=nf * (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS),
                      n_frames=nf, airtime=nf * frame_time_s())


In [ ]:
# Cell 9: Detection cache and latency
# The detector runs once per image; all channel experiments reuse the cached detections.


def build_det_cache(model, img_paths, lbl_dir, cache_path, imgsz=IMGSZ, jpeg_qualities=(25, 50, 75), warmup=20):
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
    for p in img_paths[:warmup]:
        im = cv2.imread(p)
        if im is not None:
            get_yolo_preds(model, im, imgsz=imgsz)
    items = []
    for i, p in enumerate(tqdm(img_paths, desc=os.path.basename(cache_path))):
        img = cv2.imread(p)
        if img is None:
            continue
        h, w = img.shape[:2]
        gb, gc = load_gt(os.path.join(lbl_dir, os.path.splitext(os.path.basename(p))[0] + '.txt'), w, h)
        if device == 'cuda':
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        b, s, c = get_yolo_preds(model, img, imgsz=imgsz)
        if device == 'cuda':
            torch.cuda.synchronize()
        it = dict(idx=i, path=p, w=w, h=h, gt_b=gb, gt_c=gc, det_b=b, det_s=s, det_c=c,
                  t_det=time.perf_counter() - t0)
        for q in jpeg_qualities:
            tj = time.perf_counter()
            _, buf = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), q])
            it[f'jpeg{q}_t_enc'] = time.perf_counter() - tj
            it[f'jpeg{q}_bits'] = len(buf) * 8
            it[f'jpeg{q}_det'] = get_yolo_preds(model, cv2.imdecode(buf, cv2.IMREAD_COLOR), imgsz=imgsz)
        items.append(it)
    with open(cache_path, 'wb') as f:
        pickle.dump(items, f)
    return items


_GT_MEMO = {}


def gts_of(cache):
    key = id(cache)
    if key not in _GT_MEMO:
        _GT_MEMO[key] = [make_pred_dict(it['gt_b'], np.ones(len(it['gt_b'])), it['gt_c']) for it in cache]
    return _GT_MEMO[key]


def run_condition(cache, method, loss_model, seed, fmt='fixed_bytealigned', nc=NC, order='as_is',
                  max_frames=None, legacy=False, q_pos=Q_POS, jpeg_q=50, keep_preds=False):
    preds, acc = [], []
    for it in cache:
        st = None
        if method == 'oracle':
            b, s, c = it['det_b'], it['det_s'], it['det_c']
        elif method == 'proposed':
            b, s, c, st = transmit_descriptors(
                it['det_b'], it['det_s'], it['det_c'], it['w'], it['h'], fmt, nc, loss_model,
                per_image_rng(seed, it['idx'], TAG_SEM), order, per_image_rng(seed, it['idx'], TAG_ORDER),
                max_frames, legacy, q_pos)
            st['img_lost'] = bool(st['n_det'] > 0 and st['n_recv'] == 0)
        elif method == 'jpeg':
            lost, st = transmit_image(it[f'jpeg{jpeg_q}_bits'], loss_model,
                                      per_image_rng(seed, it['idx'], TAG_JPEG), legacy)
            st['img_lost'] = bool(lost)
            b, s, c = (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)) if lost else it[f'jpeg{jpeg_q}_det']
        else:
            raise ValueError(method)
        preds.append(make_pred_dict(b, s, c))
        if st:
            acc.append(st)
    m = compute_map_full(preds, gts_of(cache))
    row = dict(method=method, fmt=fmt if method == 'proposed' else '', seed=seed, **m)
    if acc:
        a = pd.DataFrame(acc)
        row.update(info_kbits=a.info_bits.mean() / 1000, chan_kbits=a.chan_bits.mean() / 1000,
                   frames=a.n_frames.mean(), airtime_s=a.airtime.mean(), airtime_max_s=a.airtime.max(),
                   img_outage_rate=a.img_lost.mean(), max_consec_img_outage=longest_run(a.img_lost.values))
    return (row, preds) if keep_preds else row


def agg(df, keys, metrics=('map50', 'map50_95')):
    # Mean and standard deviation across seeds
    g = df.groupby(list(keys))
    out = g[list(metrics)].agg(['mean', 'std'])
    out.columns = [f'{a}_{b}' for a, b in out.columns]
    other = [c for c in ['info_kbits', 'chan_kbits', 'frames', 'airtime_s'] if c in df.columns]
    if other:
        out = out.join(g[other].mean())
    return out.reset_index()


def text_bits_per_det(cache):
    # Mean size of one detection written as a standard YOLO label line, as a conventional system would send it
    lens = []
    for it in cache:
        w, h = it['w'], it['h']
        for (x1, y1, x2, y2), sc, cl in zip(it['det_b'], it['det_s'], it['det_c']):
            line = f'{cl} {(x1 + x2) / 2 / w:.6f} {(y1 + y2) / 2 / h:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f} {sc:.4f}\n'
            lens.append(len(line))
    return int(round(8 * np.mean(lens))) if lens else 8 * 46


cache_urpc = build_det_cache(detector, VAL_FILES, VAL_LBL, f'{OUT_DIR}/cache_urpc_n640.pkl')
TEXT_BITS_PER_DET = text_bits_per_det(cache_urpc)
NUMBERS = {}  # key results, saved at the end as results_summary.json
_lat = np.array([it['t_det'] for it in cache_urpc]) * 1000
NUMBERS['t4_latency_ms_median'] = float(np.median(_lat))
NUMBERS['t4_latency_ms_mean'] = float(np.mean(_lat))
NUMBERS['t4_latency_ms_std'] = float(np.std(_lat))
NUMBERS['t4_latency_ms_p95'] = float(np.percentile(_lat, 95))
NUMBERS['mean_detections_per_image'] = float(np.mean([len(it['det_b']) for it in cache_urpc]))
NUMBERS['text_bits_per_detection'] = TEXT_BITS_PER_DET
spec('T4_LATENCY_MS', round(NUMBERS['t4_latency_ms_median'], 2), 'ms/image', 'Edge platform',
     'Median over the validation set after 20 warm-up images', 'measured')
print('Cached', len(cache_urpc), 'images. Mean detections per image:', round(NUMBERS['mean_detections_per_image'], 2))
print(f"T4 latency median {NUMBERS['t4_latency_ms_median']:.2f} ms (std {NUMBERS['t4_latency_ms_std']:.2f}). "
      "")
print('Conventional text descriptor:', TEXT_BITS_PER_DET, 'bits per detection')


In [ ]:
# Cell 10: Packet loss sweep over seeds, one descriptor per packet
if RUN_LEGACY_REPRO:
    rows = []
    for rho in RHO_LIST:
        lm = {'type': 'iid', 'rho': rho}
        rows.append(dict(run_condition(cache_urpc, 'oracle', lm, 42), rho=rho))
        for sd in tqdm(LOSS_SEEDS, desc=f'rho {rho}'):
            rows.append(dict(run_condition(cache_urpc, 'proposed', lm, sd, legacy=True), rho=rho))
            rows.append(dict(run_condition(cache_urpc, 'jpeg', lm, sd, legacy=True), rho=rho))
    df_legacy = pd.DataFrame(rows)
    t = agg(df_legacy, ['method', 'rho'])
    t.to_csv(f'{OUT_DIR}/legacy_grid_mean_std.csv', index=False)
    print(t.round(4).to_string(index=False))
    r = t[(t.method == 'proposed') & (t.rho == 0.10)].iloc[0]
    NUMBERS['legacy_proposed_rho010_mean'], NUMBERS['legacy_proposed_rho010_std'] = r.map50_mean, r.map50_std
    r = t[(t.method == 'jpeg') & (t.rho == 0.10)].iloc[0]
    NUMBERS['legacy_jpeg_rho010_mean'], NUMBERS['legacy_jpeg_rho010_std'] = r.map50_mean, r.map50_std


In [ ]:
# Cell 11: JSCC autoencoder baseline


class ImgDataset(Dataset):
    # Training images are resized once and kept in memory
    def __init__(self, img_dir, hw):
        self.paths = sorted(glob.glob(os.path.join(img_dir, '*.*'))); self.hw = hw
        self.arr = None
        if hw <= 256:
            cache = f'{WORK_DIR}/train_images_{hw}px.npy'
            if os.path.exists(cache):
                self.arr = np.load(cache, mmap_mode='r')
            else:
                from concurrent.futures import ThreadPoolExecutor
                arr = np.zeros((len(self.paths), hw, hw, 3), np.uint8)
                with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
                    for i, im in enumerate(tqdm(ex.map(self._load, self.paths), total=len(self.paths),
                                                desc=f'Caching {len(self.paths)} training images at {hw}px')):
                        arr[i] = im
                np.save(cache, arr)
                self.arr = arr

    def _load(self, p):
        img = cv2.imread(p)
        if img is None:
            return np.zeros((self.hw, self.hw, 3), np.uint8)
        return cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (self.hw, self.hw))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = self.arr[i] if self.arr is not None else self._load(self.paths[i])
        return torch.from_numpy(np.asarray(img) / 255.0).float().permute(2, 0, 1)


def erase_chunks(z, rho, sym_per_frame):
    # Zeros whole frames of channel symbols, the same erasure unit used at evaluation time
    if rho <= 0:
        return z
    B = z.shape[0]; flat = z.reshape(B, -1); n = flat.shape[1]
    nf = int(math.ceil(n / sym_per_frame))
    keep = (torch.rand(B, nf, device=z.device) >= rho).float()
    keep = keep.repeat_interleave(sym_per_frame, 1)[:, :n]
    return (flat * keep).view_as(z)


class JSCCModel(nn.Module):
    def __init__(self, C=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 5, 2, 2), nn.LeakyReLU(0.1, True),
            nn.Conv2d(32, 64, 5, 2, 2), nn.LeakyReLU(0.1, True),
            nn.Conv2d(64, C, 5, 2, 2))
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(C, 64, 4, 2, 1), nn.LeakyReLU(0.1, True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.LeakyReLU(0.1, True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())

    def channel(self, z, snr_db, rho):
        z = z / (z.pow(2).mean().sqrt() + 1e-9)
        z = erase_chunks(z, rho, FRAME_PAYLOAD_BITS // JSCC_BITS_PER_SYMBOL)
        return z + torch.randn_like(z) * (10 ** (-snr_db / 10)) ** 0.5

    def forward(self, x, snr_db=20, rho=0.0):
        return self.dec(self.channel(self.enc(x), snr_db, rho))


class DeepJSCC(nn.Module):
    # Deep JSCC (Bourtsoulatze et al.), PReLU encoder and decoder, per image power normalization
    def __init__(self, C=16):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 5, 2, 2), nn.PReLU(), nn.Conv2d(32, 64, 5, 2, 2), nn.PReLU(),
            nn.Conv2d(64, 64, 5, 2, 2), nn.PReLU(), nn.Conv2d(64, 64, 5, 1, 2), nn.PReLU(),
            nn.Conv2d(64, C, 5, 1, 2))
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(C, 64, 5, 1, 2), nn.PReLU(), nn.ConvTranspose2d(64, 64, 5, 1, 2), nn.PReLU(),
            nn.ConvTranspose2d(64, 64, 4, 2, 1), nn.PReLU(), nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.PReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid())

    def channel(self, z, snr_db, rho):
        B = z.shape[0]; flat = z.reshape(B, -1); k = flat.shape[1]
        flat = flat * math.sqrt(k) / (flat.norm(dim=1, keepdim=True) + 1e-9)
        z = erase_chunks(flat.view_as(z), rho, FRAME_PAYLOAD_BITS // JSCC_BITS_PER_SYMBOL)
        return z + torch.randn_like(z) * (10 ** (-snr_db / 10)) ** 0.5

    def forward(self, x, snr_db=20, rho=0.0):
        return self.dec(self.channel(self.enc(x), snr_db, rho))


def train_jscc(model, hw, epochs, path, snr_choices, rho_choices=(0.0, 0.05, 0.10), seed=GLOBAL_SEED):
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device)); return model.eval()
    if device != 'cuda':
        raise RuntimeError('No GPU in this session. Turn on Settings -> Accelerator -> GPU T4 x2 (or P100) and '
                           'restart; training on the CPU would take many hours.')
    set_all_seeds(seed)
    ds = ImgDataset(TRAIN_IMG, hw)
    g = torch.Generator().manual_seed(seed)
    dl = DataLoader(ds, batch_size=32 if hw <= 256 else 8, shuffle=True,
                    num_workers=0 if ds.arr is not None else 2, pin_memory=True, generator=g,
                    worker_init_fn=lambda w: np.random.seed(seed + w))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs // 2.5), gamma=0.5)
    rs = np.random.RandomState(seed)
    t_start = time.time()
    print(f'Training {os.path.basename(path)}: {epochs} epochs, {len(dl)} batches per epoch')
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x in tqdm(dl, desc=f'epoch {ep + 1}/{epochs}', leave=False):
            x = x.to(device, non_blocking=True)
            loss = F.mse_loss(model(x, float(rs.choice(snr_choices)), float(rs.choice(rho_choices))), x)
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
        sch.step()
        el = time.time() - t_start
        print(f'  epoch {ep + 1}/{epochs}  MSE {tot / len(dl):.6f}  '
              f'elapsed {el / 60:.1f} min  remaining about {el / (ep + 1) * (epochs - ep - 1) / 60:.1f} min', flush=True)
    torch.save(model.state_dict(), path)
    return model.eval()


def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6


@torch.no_grad()
def run_jscc_condition(model, cache, hw, snr_db, loss_model, seed, label):
    # Analog symbols see AWGN; erasure is applied frame by frame
    preds = []
    spf = FRAME_PAYLOAD_BITS // JSCC_BITS_PER_SYMBOL
    lm = loss_model if loss_model['type'] in ('iid', 'ge') else {'type': 'none'}
    n_sym = None
    for it in cache:
        img = cv2.imread(it['path'])
        x = torch.from_numpy(cv2.cvtColor(cv2.resize(img, (hw, hw)), cv2.COLOR_BGR2RGB)).float().div(255)
        x = x.permute(2, 0, 1).unsqueeze(0).to(device)
        z = model.enc(x)
        flat = z.flatten(); n_sym = flat.numel()
        flat = flat * math.sqrt(n_sym) / (flat.norm() + 1e-9)
        nf = int(math.ceil(n_sym / spf))
        lost = draw_frame_losses(nf, lm, per_image_rng(seed, it['idx'], TAG_JSCC))
        keep = torch.from_numpy(np.repeat(~lost, spf)[:n_sym].astype(np.float32)).to(device)
        gen = torch.Generator(device=device).manual_seed(int(seed) * 1_000_003 + it['idx'])
        noise = torch.randn(flat.shape, generator=gen, device=device) * (10 ** (-snr_db / 10)) ** 0.5
        rec = model.dec((flat * keep + noise).view_as(z))
        rec = (rec.squeeze().permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        rec = cv2.cvtColor(cv2.resize(rec, (it['w'], it['h'])), cv2.COLOR_RGB2BGR)
        preds.append(make_pred_dict(*get_yolo_preds(detector, rec)))
    m = compute_map_full(preds, gts_of(cache))
    bits = n_sym * JSCC_BITS_PER_SYMBOL
    return dict(method=label, fmt='', seed=seed, **m, channel_uses=n_sym, info_kbits=bits / 1000,
                chan_kbits=int(math.ceil(n_sym / spf)) * (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS) / 1000,
                frames=int(math.ceil(n_sym / spf)), airtime_s=int(math.ceil(n_sym / spf)) * frame_time_s())


jscc_models = {}
if RUN_JSCC:
    jscc_models['jscc256'] = (train_jscc(JSCCModel(JSCC_C).to(device), JSCC_IMG_HW, JSCC_EPOCHS,
                                         f'{OUT_DIR}/jscc256.pt', JSCC_TRAIN_SNRS), JSCC_IMG_HW)
    NUMBERS['jscc256_params_M'] = count_params(jscc_models['jscc256'][0])


In [ ]:
# Cell 12: DeepJSCC baseline
if RUN_DEEPJSCC:
    jscc_models['deepjscc256'] = (train_jscc(DeepJSCC(DEEPJSCC_C).to(device), JSCC_IMG_HW, DEEPJSCC_EPOCHS,
                                             f'{OUT_DIR}/deepjscc256.pt', JSCC_TRAIN_SNRS), JSCC_IMG_HW)
    NUMBERS['deepjscc_params_M'] = count_params(jscc_models['deepjscc256'][0])
    print('DeepJSCC ready. Parameters (M):', round(NUMBERS['deepjscc_params_M'], 3))

    # Reconstruction PSNR of both learned baselines on validation images
    @torch.no_grad()
    def psnr_db(model, hw, snr_db, n=100):
        vals = []
        for it in cache_urpc[:n]:
            im = cv2.cvtColor(cv2.resize(cv2.imread(it['path']), (hw, hw)), cv2.COLOR_BGR2RGB)
            x = torch.from_numpy(im).float().div(255).permute(2, 0, 1)[None].to(device)
            mse = F.mse_loss(model(x, snr_db, 0.0), x).item()
            vals.append(10 * math.log10(1.0 / max(mse, 1e-10)))
        return float(np.mean(vals))
    psnr_rows = []
    for label, (mdl, hw) in jscc_models.items():
        mdl.eval()
        for snr in [0, 10, 20]:
            psnr_rows.append(dict(model=label, snr_db=snr, psnr_db=psnr_db(mdl, hw, snr)))
    df_psnr = pd.DataFrame(psnr_rows).pivot(index='model', columns='snr_db', values='psnr_db')
    df_psnr.to_csv(f'{OUT_DIR}/jscc_psnr_check.csv')
    print('Reconstruction PSNR (dB) on 100 validation images:'); print(df_psnr.round(2).to_string())


In [ ]:
# Cell 13: JSCC at 640 x 640 (session D)
if RUN_JSCC640:
    jscc_models['jscc640'] = (train_jscc(JSCCModel(JSCC640_C).to(device), 640, JSCC640_EPOCHS,
                                         f'{OUT_DIR}/jscc640.pt', JSCC_TRAIN_SNRS), 640)
    print('JSCC 640 latent channel uses:', JSCC640_C * 80 * 80)


In [ ]:
# Cell 14: SNR sweep with bit errors
if RUN_BER_SWEEP:
    rows = []
    for snr in tqdm(SNR_LIST_BER, desc='SNR'):
        lm = {'type': 'ber', 'snr_db': snr}
        for sd in LOSS_SEEDS:
            for fmt in ['fixed_bytealigned', 'fixed_bitpacked']:
                rows.append(dict(run_condition(cache_urpc, 'proposed', lm, sd, fmt=fmt), snr=snr))
            rows.append(dict(run_condition(cache_urpc, 'jpeg', lm, sd), snr=snr))
        for label, (mdl, hw) in jscc_models.items():
            for sd in JSCC_SEEDS:
                rows.append(dict(run_jscc_condition(mdl, cache_urpc, hw, snr, lm, sd, label), snr=snr))
    df_ber = pd.DataFrame(rows)
    df_ber['key'] = df_ber.method + df_ber.fmt.radd('_').where(df_ber.fmt != '', '')
    t = agg(df_ber, ['key', 'snr'])
    t['frame_error_rate'] = t.snr.map(frame_error_rate)
    t.to_csv(f'{OUT_DIR}/ber_sweep_mean_std.csv', index=False)
    print(t.round(4).to_string(index=False))

    plt.figure(figsize=(6, 4))
    for k, g in t.groupby('key'):
        plt.errorbar(g.snr, g.map50_mean, yerr=g.map50_std.fillna(0), marker='o', capsize=3, label=k)
    plt.xlabel('SNR Es/N0 (dB)'); plt.ylabel('mAP@0.5')
    plt.title(f'mAP vs SNR, {MODULATION.upper()} {FADING}, {FRAME_PAYLOAD_BITS // 8} B frames')
    plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/fig_map_vs_snr_ber.png', dpi=300); plt.show()
    for k in t.key.unique():
        g = t[t.key == k].set_index('snr')
        NUMBERS[f'ber_{k}_snr{min(SNR_LIST_BER)}'] = g.map50_mean.iloc[0]
        NUMBERS[f'ber_{k}_snr{max(SNR_LIST_BER)}'] = g.map50_mean.iloc[-1]
        if 10 in g.index:
            NUMBERS[f'ber_{k}_snr10_mean'] = g.loc[10, 'map50_mean']
            NUMBERS[f'ber_{k}_snr10_std'] = g.loc[10, 'map50_std']


In [ ]:
# Cell 15: Burst loss (Gilbert-Elliott) versus iid loss
if RUN_BURST:
    rows = []
    loss_specs = [('iid', 1, {'type': 'iid', 'rho': GE_MEAN_LOSS})] + \
                 [('ge', L, ge_model(GE_MEAN_LOSS, L)) for L in GE_BURST_LENGTHS]
    methods = [('proposed', 'fixed_bytealigned'), ('proposed', 'fixed_bitpacked'), ('jpeg', '')]
    for name, L, lm in tqdm(loss_specs, desc='loss model'):
        for sd in LOSS_SEEDS:
            for meth, fmt in methods:
                kw = {'fmt': fmt} if fmt else {}
                r = run_condition(cache_urpc, meth, lm, sd, **kw)
                r.update(model=name, burst=L, channel='per_image', max_lost_frame_run=np.nan, max_outage_s=np.nan)
                rows.append(r)
                if name == 'ge':
                    chan = GEStream(lm, np.random.RandomState([sd, 31337, 0 if meth == 'jpeg' else 1]))
                    r = run_condition(cache_urpc, meth, {'type': 'stream', 'chan': chan}, sd, **kw)
                    run_len = chan.longest_loss_run()
                    r.update(model=name, burst=L, channel='continuous', max_lost_frame_run=run_len,
                             max_outage_s=run_len * frame_time_s())
                    rows.append(r)
    df_burst = pd.DataFrame(rows)
    keys = ['method', 'fmt', 'model', 'burst', 'channel']
    t = agg(df_burst, keys, metrics=('map50', 'map50_95', 'img_outage_rate'))
    extra = df_burst.groupby(keys)[['max_consec_img_outage', 'max_lost_frame_run', 'max_outage_s']].mean().reset_index()
    t = t.merge(extra, on=keys)
    t.to_csv(f'{OUT_DIR}/burst_loss_mean_std.csv', index=False)
    print(t.round(4).to_string(index=False))


In [ ]:
# Cell 16: Descriptor format ablation and ARQ delivery
if RUN_FORMAT_ABLATION:
    rows = []
    for fmt in FORMATS:
        for lm_name, lm in [('no_loss', {'type': 'none'}), ('iid_0.10', {'type': 'iid', 'rho': 0.10})]:
            for sd in (LOSS_SEEDS if lm['type'] != 'none' else [42]):
                rows.append(dict(run_condition(cache_urpc, 'proposed', lm, sd, fmt=fmt), loss=lm_name))
    for sd in LOSS_SEEDS:
        rows.append(dict(run_condition(cache_urpc, 'jpeg', {'type': 'iid', 'rho': 0.10}, sd), loss='iid_0.10'))
    df_fmt = pd.DataFrame(rows)
    t = agg(df_fmt, ['method', 'fmt', 'loss'])
    t['bits_per_det'] = t.fmt.map(lambda f: FORMATS[f](NC) if f else np.nan)
    t['max_fps_at_link'] = 1.0 / t.airtime_s
    t.to_csv(f'{OUT_DIR}/descriptor_format_ablation.csv', index=False)
    print(t.round(4).to_string(index=False))
    kb = t[t.loss == 'iid_0.10'].set_index(['method', 'fmt']).info_kbits
    jpeg_kb = kb[('jpeg', '')]
    NUMBERS['gain_onboard_inference_x'] = jpeg_kb / kb[('proposed', 'float32')]
    NUMBERS['gain_encoding_x'] = kb[('proposed', 'float32')] / kb[('proposed', 'fixed_bitpacked')]
    NUMBERS['gain_vs_text_x'] = kb[('proposed', 'text_yolo')] / kb[('proposed', 'fixed_bitpacked')]
    NUMBERS['bitpacked_kbits_per_img'] = kb[('proposed', 'fixed_bitpacked')]
    NUMBERS['bitpacked_kbps_at_10fps'] = 10 * kb[('proposed', 'fixed_bitpacked')]
    print(f"Onboard inference gain {NUMBERS['gain_onboard_inference_x']:.1f}x, "
          f"encoding gain {NUMBERS['gain_encoding_x']:.2f}x")
    # Proof that the bit packed size is real, pack and unpack one image
    it = max(cache_urpc, key=lambda x: len(x['det_b']))
    data, nbits = pack_bitpacked(it['det_b'], it['det_s'], it['det_c'], it['w'], it['h'], NC)
    ub, us, uc = unpack_bitpacked(data, len(it['det_b']), NC, it['w'], it['h'])
    eb, es, ec = encode_decode('fixed_bitpacked', it['det_b'], it['det_s'], it['det_c'], it['w'], it['h'])
    print('Bit packing round trip exact:', np.allclose(ub, eb, atol=1e-3) and np.array_equal(uc, ec),
          '| bits', nbits, 'for', len(it['det_b']), 'detections')

    # Reliable delivery with selective repeat ARQ at the same 10 percent frame loss
    FER_OP = 0.10
    arq_rows = []
    for label, frames in [
            ('proposed_fixed_bitpacked', [max(1, math.ceil(len(it['det_b']) / (FRAME_PAYLOAD_BITS // FORMATS['fixed_bitpacked'](NC))))
                                          for it in cache_urpc]),
            ('proposed_fixed_bytealigned', [max(1, math.ceil(len(it['det_b']) / (FRAME_PAYLOAD_BITS // FORMATS['fixed_bytealigned'](NC))))
                                            for it in cache_urpc]),
            ('jpeg_q50', [math.ceil(it['jpeg50_bits'] / FRAME_PAYLOAD_BITS) for it in cache_urpc])]:
        times = np.array([arq_delivery_time(n, FER_OP) for n in frames])
        arq_rows.append(dict(method=label, frame_loss=FER_OP, mean_frames=np.mean(frames),
                             mean_delivery_s=times.mean(), p95_delivery_s=np.percentile(times, 95),
                             delivered_fps=1.0 / times.mean(), rtt_s=2 * OPERATING_RANGE_M / SOUND_SPEED_MPS))
    free_prop = run_condition(cache_urpc, 'proposed', {'type': 'none'}, 42, fmt='fixed_bitpacked')['map50']
    free_jpeg = run_condition(cache_urpc, 'jpeg', {'type': 'none'}, 42)['map50']
    df_arq = pd.DataFrame(arq_rows)
    df_arq['map50_when_delivered'] = [free_prop, free_prop, free_jpeg]
    df_arq.to_csv(f'{OUT_DIR}/arq_reliable_delivery.csv', index=False)
    print(df_arq.round(4).to_string(index=False))
    NUMBERS['jpeg_arq_delivery_s'] = float(df_arq.mean_delivery_s.iloc[2])
    NUMBERS['jpeg_arq_fps'] = float(df_arq.delivered_fps.iloc[2])
    NUMBERS['proposed_arq_fps'] = float(df_arq.delivered_fps.iloc[0])


In [ ]:
# Cell 17: Quantization ablation
if RUN_QUANT_ABLATION:
    rows = []
    for q in QUANT_LEVELS:
        for sd in LOSS_SEEDS:
            rows.append(dict(run_condition(cache_urpc, 'proposed', {'type': 'iid', 'rho': 0.05}, sd,
                                           legacy=True, q_pos=q), q_pos=q))
    t1 = agg(pd.DataFrame(rows), ['q_pos'])
    t1.to_csv(f'{OUT_DIR}/quantization_ablation_legacy_mean_std.csv', index=False)
    NUMBERS['quant_spread_map50_legacy'] = float(t1.map50_mean.max() - t1.map50_mean.min())

    rows = []
    for q in QUANT_LEVELS:
        rows.append(dict(run_condition(cache_urpc, 'proposed', {'type': 'none'}, 42, fmt='fixed_bitpacked',
                                       q_pos=q), q_pos=q, cond='no_loss'))
        for sd in LOSS_SEEDS:
            rows.append(dict(run_condition(cache_urpc, 'proposed', {'type': 'iid', 'rho': 0.10}, sd,
                                           fmt='fixed_bitpacked', q_pos=q), q_pos=q, cond='iid_0.10'))
    t2 = agg(pd.DataFrame(rows), ['cond', 'q_pos'])
    t2['bits_per_det'] = t2.q_pos.map(lambda q: FORMATS['fixed_bitpacked'](NC, q))
    t2['grid_step_px_at_640'] = IMGSZ / t2.q_pos
    t2['max_fps_at_link'] = 1.0 / (t2.frames * frame_time_s())
    t2.to_csv(f'{OUT_DIR}/quantization_ablation_mean_std.csv', index=False)
    print('Byte aligned, one descriptor per packet, 5 percent loss')
    print(t1.round(4).to_string(index=False))
    print('Bit packed, real bit cost')
    print(t2.round(4).to_string(index=False))
    lossy = t2[t2.cond == 'iid_0.10']
    NUMBERS['quant_spread_map50'] = float(lossy.map50_mean.max() - lossy.map50_mean.min())
    print('mAP@0.5 spread across grids at 10 percent loss:', round(NUMBERS['quant_spread_map50'], 4))


In [ ]:
# Cell 17B: Confidence threshold and JPEG quality ablations
# Higher thresholds are applied as filters on the cached detections (cache built at 0.10).
if RUN_QUANT_ABLATION:
    FER_OP = 0.10
    ft = frame_time_s()

    # Part 1, confidence threshold, bit packed descriptors, 10 percent iid frame loss
    rows = []
    for thr in [0.10, 0.25, 0.50]:
        cache_thr = []
        for it in cache_urpc:
            keep = it['det_s'] >= thr
            it2 = dict(it)
            it2['det_b'], it2['det_s'], it2['det_c'] = it['det_b'][keep], it['det_s'][keep], it['det_c'][keep]
            cache_thr.append(it2)
        n_mean = float(np.mean([len(it['det_b']) for it in cache_thr]))
        free = run_condition(cache_thr, 'proposed', {'type': 'none'}, 42, fmt='fixed_bitpacked')
        rows.append(dict(free, thr=thr, cond='no_loss', n_mean=n_mean))
        for sd in LOSS_SEEDS:
            r = run_condition(cache_thr, 'proposed', {'type': 'iid', 'rho': FER_OP}, sd, fmt='fixed_bitpacked')
            rows.append(dict(r, thr=thr, cond='iid_0.10', n_mean=n_mean))
    t_conf = agg(pd.DataFrame(rows), ['cond', 'thr'])
    t_conf = t_conf.merge(pd.DataFrame(rows)[['thr', 'n_mean']].drop_duplicates(), on='thr')
    t_conf['img_per_s'] = 1.0 / t_conf.airtime_s
    t_conf.to_csv(f'{OUT_DIR}/confidence_threshold_ablation.csv', index=False)
    print(t_conf.round(4).to_string(index=False))

    # Part 2, JPEG quality, accuracy without loss and reliable delivery with ARQ at 10 percent frame loss
    rows = []
    for q in [25, 50, 75]:
        free = run_condition(cache_urpc, 'jpeg', {'type': 'none'}, 42, jpeg_q=q)
        frames = [math.ceil(it[f'jpeg{q}_bits'] / FRAME_PAYLOAD_BITS) for it in cache_urpc]
        t_arq = np.array([arq_delivery_time(n, FER_OP) for n in frames])
        lossy = [run_condition(cache_urpc, 'jpeg', {'type': 'iid', 'rho': FER_OP}, sd, jpeg_q=q)['map50']
                 for sd in LOSS_SEEDS]
        rows.append(dict(jpeg_q=q, map50=free['map50'], map50_95=free['map50_95'],
                         info_kbits=free['info_kbits'], chan_kbits=free['chan_kbits'],
                         frames=np.mean(frames), airtime_s=np.mean(frames) * ft,
                         arq_delivery_s=t_arq.mean(), arq_img_per_s=1.0 / t_arq.mean(),
                         map50_iid010_no_arq=float(np.mean(lossy))))
    t_jpeg = pd.DataFrame(rows)
    t_jpeg.to_csv(f'{OUT_DIR}/jpeg_quality_ablation.csv', index=False)
    print(t_jpeg.round(4).to_string(index=False))


In [ ]:
# Cell 18: Per class AP under loss and truncation
if RUN_PER_CLASS:
    rows, rp_rows = [], []
    for rho in [0.0, 0.05, 0.10, 0.20, 0.30]:
        for sd in (LOSS_SEEDS if rho > 0 else [42]):
            row, preds = run_condition(cache_urpc, 'proposed', {'type': 'iid', 'rho': rho}, sd,
                                       legacy=True, keep_preds=True)
            pc = compute_per_class_ap(preds, gts_of(cache_urpc), CLASS_NAMES)
            rows.append(dict(rho=rho, seed=sd, **pc))
            # Recall and precision split per class at the operating threshold
            for k, nm in enumerate(CLASS_NAMES):
                tp = fp = ngt = 0
                for it, pr in zip(cache_urpc, preds):
                    pb, ps, pcl = pr['boxes'].numpy(), pr['scores'].numpy(), pr['labels'].numpy()
                    flag = match_tp(pb, ps, pcl, it['gt_b'], it['gt_c'])
                    tp += int((flag & (pcl == k)).sum()); fp += int((~flag & (pcl == k)).sum())
                    ngt += int((it['gt_c'] == k).sum())
                rp_rows.append(dict(rho=rho, seed=sd, cls=nm, recall=tp / max(ngt, 1),
                                    precision=tp / max(tp + fp, 1)))
    pc_df = pd.DataFrame(rows).groupby('rho')[CLASS_NAMES].agg(['mean', 'std'])
    pc_df.to_csv(f'{OUT_DIR}/per_class_ap_vs_rho.csv')
    rp = pd.DataFrame(rp_rows).groupby(['cls', 'rho'])[['recall', 'precision']].mean().reset_index()
    rp.to_csv(f'{OUT_DIR}/per_class_recall_precision_vs_rho.csv', index=False)
    print(pc_df.round(4).to_string()); print(rp.round(4).to_string(index=False))
    # Share of transmitted descriptors that are false positives at the 0.10 threshold
    tp_flags = [match_tp(it['det_b'], it['det_s'], it['det_c'], it['gt_b'], it['gt_c']) for it in cache_urpc]
    tot = sum(len(f) for f in tp_flags); fpn = sum(int((~f).sum()) for f in tp_flags)
    NUMBERS['fp_share_of_descriptors'] = fpn / max(tot, 1)
    print('False positive share of transmitted descriptors:', round(NUMBERS['fp_share_of_descriptors'], 3))
    plt.figure(figsize=(6, 4))
    for nm in CLASS_NAMES:
        m = pc_df[(nm, 'mean')]; s = pc_df[(nm, 'std')].fillna(0)
        plt.errorbar(m.index, m.values, yerr=s.values, marker='o', capsize=3, label=nm)
    plt.xlabel('Descriptor loss rate'); plt.ylabel('AP@0.5'); plt.grid(alpha=0.3); plt.legend()
    plt.tight_layout(); plt.savefig(f'{OUT_DIR}/fig_per_class_ap_vs_rho.png', dpi=300); plt.show()

    # Part 2, per class AP under link budget truncation confidence ordered, 10 percent frame loss
    tr_rows = []
    for fps in [10, 5]:
        budget = max(1, int((1.0 / fps) // frame_time_s()))
        for sd in LOSS_SEEDS:
            row, preds = run_condition(cache_urpc, 'proposed', {'type': 'iid', 'rho': 0.10}, sd, fmt='fixed_bitpacked',
                                       order='conf_desc', max_frames=budget, keep_preds=True)
            pc = compute_per_class_ap(preds, gts_of(cache_urpc), CLASS_NAMES)
            tr_rows.append(dict(fps=fps, frames_budget=budget, seed=sd, **pc))
    tr = pd.DataFrame(tr_rows).groupby(['fps', 'frames_budget'])[CLASS_NAMES].agg(['mean', 'std'])
    tr.to_csv(f'{OUT_DIR}/per_class_ap_truncation.csv')
    print(tr.round(4).to_string())

    # Confidence profile per class, explains which classes are cut first
    per_frame = FRAME_PAYLOAD_BITS // FORMATS['fixed_bitpacked'](NC)
    prof = []
    for k, nm in enumerate(CLASS_NAMES):
        sc_all, sc_tp, kept_tp, all_tp = [], [], 0, 0
        for it, f in zip(cache_urpc, tp_flags):
            m = it['det_c'] == k
            sc_all.extend(it['det_s'][m].tolist()); sc_tp.extend(it['det_s'][m & f].tolist())
            top = np.argsort(-it['det_s'], kind='stable')[:per_frame]
            keep = np.zeros(len(f), bool); keep[top] = True
            kept_tp += int((m & f & keep).sum()); all_tp += int((m & f).sum())
        prof.append(dict(cls=nm, descriptors=len(sc_all), mean_score=np.mean(sc_all) if sc_all else np.nan,
                         mean_score_true_pos=np.mean(sc_tp) if sc_tp else np.nan,
                         true_pos_kept_in_one_frame=kept_tp / max(all_tp, 1)))
    df_prof = pd.DataFrame(prof)
    df_prof.to_csv(f'{OUT_DIR}/per_class_confidence_profile.csv', index=False)
    print(df_prof.round(4).to_string(index=False))


In [ ]:
# Cell 19: Link budget, frame rate and confidence ordered sending
if RUN_BUDGET:
    rows = []
    for fps in [1, 2, 5, 10]:
        budget = int((1.0 / fps) // frame_time_s())
        for fmt in ['fixed_bytealigned', 'fixed_bitpacked']:
            for order in ['conf_desc', 'random']:
                for sd in LOSS_SEEDS:
                    rows.append(dict(run_condition(cache_urpc, 'proposed', {'type': 'iid', 'rho': 0.10}, sd,
                                                   fmt=fmt, order=order, max_frames=budget),
                                     fps=fps, frames_budget=budget, order=order))
    t = agg(pd.DataFrame(rows), ['fmt', 'order', 'fps', 'frames_budget'])
    t.to_csv(f'{OUT_DIR}/link_budget_ordering.csv', index=False)
    print('Frame time (s):', round(frame_time_s(), 4)); print(t.round(4).to_string(index=False))

    # Strict feasibility per scene density
    def n_gt(it):
        return len(it['gt_b'])
    buckets = {'sparse_1_3': lambda n: 1 <= n <= 3, 'medium_4_8': lambda n: 4 <= n <= 8,
               'dense_9_plus': lambda n: n >= 9, 'all': lambda n: True}
    fr = []
    for bname, f in buckets.items():
        items = [it for it in cache_urpc if f(n_gt(it))]
        ndet = np.array([len(it['det_b']) for it in items])
        for fmt in FORMATS:
            per = max(1, FRAME_PAYLOAD_BITS // FORMATS[fmt](NC))
            for stat, n in [('mean', ndet.mean()), ('p95', np.percentile(ndet, 95)), ('max', ndet.max())]:
                frames = max(1, math.ceil(n / per))
                fr.append(dict(bucket=bname, fmt=fmt, stat=stat, detections=n,
                               info_kbits=(HEADER_BITS + FORMATS[fmt](NC) * n) / 1000,
                               frames=frames, max_fps=1.0 / (frames * frame_time_s())))
    df_feas = pd.DataFrame(fr)
    df_feas.to_csv(f'{OUT_DIR}/strict_feasibility_by_scene.csv', index=False)
    print(df_feas.round(3).to_string(index=False))
    d = df_feas[(df_feas.bucket == 'dense_9_plus') & (df_feas.stat == 'mean')].set_index('fmt')
    NUMBERS['dense_max_fps_bytealigned'] = d.loc['fixed_bytealigned', 'max_fps']
    NUMBERS['dense_max_fps_bitpacked'] = d.loc['fixed_bitpacked', 'max_fps']

    # Preamble sensitivity and compute ceiling of the edge board
    ndet = np.array([len(it['det_b']) for it in cache_urpc])
    sens = []
    for pre in [0.0, 0.0125, 0.025, 0.05, 0.1]:
        for fmt in ['fixed_bytealigned', 'fixed_bitpacked']:
            per = max(1, FRAME_PAYLOAD_BITS // FORMATS[fmt](NC))
            air = np.maximum(1, np.ceil(ndet / per)).mean() * frame_time_s(pre)
            sens.append(dict(preamble_s=pre, fmt=fmt, mean_airtime_s=air, link_fps=1 / air,
                             pi5_compute_fps=1000.0 / PI5_LATENCY_MS,
                             achievable_fps=min(1 / air, 1000.0 / PI5_LATENCY_MS)))
    df_sens = pd.DataFrame(sens)
    df_sens.to_csv(f'{OUT_DIR}/fps_preamble_sensitivity.csv', index=False)
    print(df_sens.round(3).to_string(index=False))
    r = df_sens[(df_sens.preamble_s == FRAME_PREAMBLE_S)].set_index('fmt')
    NUMBERS['mean_fps_bitpacked'] = float(r.loc['fixed_bitpacked', 'achievable_fps'])
    NUMBERS['mean_fps_bytealigned'] = float(r.loc['fixed_bytealigned', 'achievable_fps'])
    NUMBERS['pi5_compute_fps'] = 1000.0 / PI5_LATENCY_MS
    print(f"Frame rate: {NUMBERS['mean_fps_bitpacked']:.1f} fps on average with bit packing "
          f"({NUMBERS['mean_fps_bytealigned']:.1f} fps byte aligned); 10 fps for sparse scenes or with "
          f"confidence-ordered truncation. The Raspberry Pi 5 computes {NUMBERS['pi5_compute_fps']:.1f} fps, "
          f"so the link, not the board, is the bottleneck.")


In [ ]:
# Cell 20: Resolution matched controls
if RUN_RES_CONTROL:
    cache_256 = build_det_cache(detector, VAL_FILES, VAL_LBL, f'{OUT_DIR}/cache_urpc_n256.pkl',
                                imgsz=256, jpeg_qualities=())
    rows = [dict(run_condition(cache_256, 'oracle', {'type': 'none'}, 42), input_px=256),
            dict(run_condition(cache_urpc, 'oracle', {'type': 'none'}, 42), input_px=640)]
    for sd in LOSS_SEEDS:
        for c_, px in [(cache_256, 256), (cache_urpc, 640)]:
            rows.append(dict(run_condition(c_, 'proposed', {'type': 'iid', 'rho': 0.10}, sd), input_px=px))
    for label, (mdl, hw) in jscc_models.items():
        for sd in JSCC_SEEDS:
            rows.append(dict(run_jscc_condition(mdl, cache_urpc, hw, 10, {'type': 'iid', 'rho': 0.10}, sd, label),
                             input_px=hw))
    t = agg(pd.DataFrame(rows), ['method', 'input_px'])
    t.to_csv(f'{OUT_DIR}/resolution_controls.csv', index=False)
    print(t.round(4).to_string(index=False))


In [ ]:
# Cell 21: Split computing baseline
SPLIT_LAYERS = [4, 6, 9]


def detection_modules(yolo, probe_path):
    # Layer list actually run by the predictor (AutoBackend in newer Ultralytics versions)
    if getattr(yolo, 'predictor', None) is None:
        yolo(cv2.imread(probe_path), verbose=False)
    cands = []
    p = getattr(yolo, 'predictor', None)
    if p is not None and getattr(p, 'model', None) is not None:
        m = p.model
        for _ in range(3):
            if isinstance(m, (nn.Sequential, nn.ModuleList)):
                break
            m = getattr(m, 'model', m)
        if isinstance(m, (nn.Sequential, nn.ModuleList)):
            cands.append(m)
    cands.append(yolo.model.model)
    return cands


class SplitQuantizer:
    def __init__(self, yolo, bits, probe_path):
        self.bits = bits; self.raw_bits = 0; self.zlib_bits = 0; self.handles = []
        for layers in detection_modules(yolo, probe_path):
            self.handles = [layers[li].register_forward_hook(self.hook) for li in SPLIT_LAYERS]
            yolo(cv2.imread(probe_path), verbose=False)
            if self.raw_bits > 0:
                break
            self.remove()
        if self.raw_bits == 0:
            raise RuntimeError('Split hooks were never called. Send the Ultralytics version printed by Cell 1.')
        self.raw_bits = self.zlib_bits = 0

    def hook(self, module, inp, out):
        mn, mx = out.min(), out.max(); L = 2 ** self.bits - 1
        q = torch.round((out - mn) / (mx - mn + 1e-9) * L)
        self.raw_bits += out.numel() * self.bits
        self.zlib_bits += len(zlib.compress(q.to(torch.uint8).cpu().numpy().tobytes(), 9)) * 8
        return q / L * (mx - mn) + mn

    def remove(self):
        for h in self.handles:
            h.remove()


if RUN_SPLIT:
    rows = []
    for bits in [8, 4]:
        sq = SplitQuantizer(detector, bits, cache_urpc[0]['path'])
        preds, raw, zb = [], [], []
        for it in tqdm(cache_urpc, desc=f'split {bits} bit'):
            sq.raw_bits = sq.zlib_bits = 0
            preds.append(make_pred_dict(*get_yolo_preds(detector, cv2.imread(it['path']))))
            raw.append(sq.raw_bits); zb.append(sq.zlib_bits)
        sq.remove()
        if np.mean(zb) == 0:
            raise RuntimeError('Split features were not captured; reNSKWMDKJEDIEJDIUHFsults would be invalid.')
        m = compute_map_full(preds, gts_of(cache_urpc))
        rows.append(dict(method=f'split_l4_l6_l9_{bits}bit', **m, raw_kbits=np.mean(raw) / 1000,
                         zlib_kbits=np.mean(zb) / 1000,
                         frames=math.ceil(np.mean(zb) / FRAME_PAYLOAD_BITS),
                         airtime_s=math.ceil(np.mean(zb) / FRAME_PAYLOAD_BITS) * frame_time_s()))
    df_split = pd.DataFrame(rows)
    df_split.to_csv(f'{OUT_DIR}/split_computing_baseline.csv', index=False)
    print(df_split.round(3).to_string(index=False))
    print('Accuracy at zero loss; any lost frame makes the feature set unusable.')


In [ ]:
# Cell 22: Computational complexity on the AUV side
# PUBLISHED_COMPLEXITY: optional rows with values reported by other papers; rows with None are skipped.
PUBLISHED_COMPLEXITY = [
    # dict(name='...', reference='[xx] Authors, Venue, Year', auv_params_M=..., auv_gflops=..., input_px=..., note='...'),
]


def count_conv_macs(model, x, group_of=None):
    # Returns total MACs and, if group_of maps module then group name, MACs per group
    total = {'all': 0}
    hooks = []

    def make(gname):
        def hook(m, inp, out):
            if isinstance(m, nn.ConvTranspose2d):
                n = (inp[0].numel() // inp[0].shape[0]) * (m.out_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
            else:
                n = (out.numel() // out.shape[0]) * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
            total['all'] += n
            if gname is not None:
                total[gname] = total.get(gname, 0) + n
        return hook

    for mod in model.modules():
        if isinstance(mod, (nn.Conv2d, nn.ConvTranspose2d)):
            hooks.append(mod.register_forward_hook(make(group_of.get(mod) if group_of else None)))
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    return total


@torch.no_grad()
def time_module(fn, x, reps=50):
    for _ in range(5):
        fn(x)
    if device == 'cuda':
        torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(x)
        if device == 'cuda':
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.median(ts))


COMPLEX = {}
if RUN_COMPLEXITY:
    import copy
    det = copy.deepcopy(detector.model).to(device).float().eval()
    layers = det.model
    group_of = {}
    for li, layer in enumerate(layers):
        for m in layer.modules():
            group_of[m] = 'backbone' if li <= max(SPLIT_LAYERS) else 'neck_head'
    x640 = torch.zeros(1, 3, IMGSZ, IMGSZ, device=device)
    macs = count_conv_macs(det, x640, group_of)
    yolo_gflops = 2 * macs['all'] / 1e9
    bb_gflops = 2 * macs.get('backbone', 0) / 1e9
    n_par = sum(p.numel() for p in det.parameters()) / 1e6
    bb_par = sum(p.numel() for li, l in enumerate(layers) if li <= max(SPLIT_LAYERS) for p in l.parameters()) / 1e6
    lat_t4 = NUMBERS.get('t4_latency_ms_median', np.nan)
    COMPLEX['proposed_yolov8n'] = dict(scheme='Proposed (YOLOv8-n onboard + descriptors)', auv_params_M=n_par,
                                       auv_gflops=yolo_gflops, surface_gflops=0.0, t4_auv_ms=lat_t4,
                                       pi5_auv_ms=PI5_LATENCY_MS, pi5_source='official benchmark')
    COMPLEX['split_l4_l6_l9'] = dict(scheme='Split computing (backbone onboard, layers 4/6/9 sent)', auv_params_M=bb_par,
                                     auv_gflops=bb_gflops, surface_gflops=yolo_gflops - bb_gflops,
                                     t4_auv_ms=np.nan, pi5_auv_ms=PI5_LATENCY_MS * bb_gflops / yolo_gflops,
                                     pi5_source='estimate (FLOP scaling)')
    COMPLEX['jpeg_q50'] = dict(scheme='JPEG + detection at the surface', auv_params_M=0.0, auv_gflops=0.0,
                               surface_gflops=yolo_gflops,
                               t4_auv_ms=1000 * float(np.median([it['jpeg50_t_enc'] for it in cache_urpc])),
                               pi5_auv_ms=1000 * float(np.median([it['jpeg50_t_enc'] for it in cache_urpc])),
                               pi5_source='measured JPEG encode time on the Kaggle CPU')
    for label, (mdl, hw) in jscc_models.items():
        xx = torch.zeros(1, 3, hw, hw, device=device)
        e = 2 * count_conv_macs(mdl.enc, xx)['all'] / 1e9
        z = mdl.enc(xx)
        d = 2 * count_conv_macs(mdl.dec, z)['all'] / 1e9
        COMPLEX[label] = dict(scheme=f'{label} encoder onboard, decoder + YOLOv8-n at the surface',
                              auv_params_M=sum(p.numel() for p in mdl.enc.parameters()) / 1e6, auv_gflops=e,
                              surface_gflops=d + yolo_gflops, t4_auv_ms=time_module(mdl.enc, xx),
                              pi5_auv_ms=PI5_LATENCY_MS * e / yolo_gflops, pi5_source='estimate (FLOP scaling)')
    df_cx = pd.DataFrame([dict(key=k, **v) for k, v in COMPLEX.items()])
    pub = [dict(key='published', scheme=p['name'] + ' ' + p['reference'], auv_params_M=p['auv_params_M'],
                auv_gflops=p['auv_gflops'], surface_gflops=np.nan, t4_auv_ms=np.nan, pi5_auv_ms=np.nan,
                pi5_source=p.get('note', 'reported by the authors'))
           for p in PUBLISHED_COMPLEXITY if p.get('auv_gflops') is not None]
    if pub:
        df_cx = pd.concat([df_cx, pd.DataFrame(pub)], ignore_index=True)
    df_cx['pi5_compute_fps'] = 1000.0 / df_cx.pi5_auv_ms.replace(0, np.nan)
    df_cx.to_csv(f'{OUT_DIR}/complexity_auv_side.csv', index=False)
    print(df_cx.round(3).to_string(index=False))
    NUMBERS['yolov8n_gflops_measured'] = yolo_gflops
    NUMBERS['split_backbone_gflops'] = bb_gflops


In [ ]:
# Cell 23: Energy, CAPEX and OPEX per transmission scheme
if RUN_COST:
    ft = frame_time_s()
    FER_OP = 0.10

    def frames_descr(fmt):
        per = max(1, FRAME_PAYLOAD_BITS // FORMATS[fmt](NC))
        return np.array([max(1, math.ceil(len(it['det_b']) / per)) for it in cache_urpc])

    fr_bit = frames_descr('fixed_bitpacked'); fr_byte = frames_descr('fixed_bytealigned')
    fr_jpeg = np.array([math.ceil(it['jpeg50_bits'] / FRAME_PAYLOAD_BITS) for it in cache_urpc])

    # Detection quality at the 10 percent frame loss operating point, mean over seeds
    def map_at(method, fmt=None):
        kw = {'fmt': fmt} if fmt else {}
        rr = [run_condition(cache_urpc, method, {'type': 'iid', 'rho': FER_OP}, sd, **kw) for sd in LOSS_SEEDS]
        return float(np.mean([r['map50'] for r in rr])), 1.0 - float(np.mean([r['img_outage_rate'] for r in rr]))

    (map_bit, del_bit), (map_byte, del_byte) = map_at('proposed', 'fixed_bitpacked'), map_at('proposed', 'fixed_bytealigned')
    map_jpeg, del_jpeg = map_at('jpeg')
    map_jpeg_free = run_condition(cache_urpc, 'jpeg', {'type': 'none'}, 42)['map50']
    ber_csv = f'{OUT_DIR}/ber_sweep_mean_std.csv'
    jscc_map = {}
    if os.path.exists(ber_csv):
        tb = pd.read_csv(ber_csv)
        for k in tb.key.unique():
            g = tb[tb.key == k].set_index('snr')
            if 10 in g.index:
                jscc_map[k] = float(g.loc[10, 'map50_mean'])

    def cx(key, field, default):
        return COMPLEX[key][field] if key in COMPLEX else default

    pi5_ms = PI5_LATENCY_MS
    schemes = [
        # name, board, compute ms on that board, mean airtime s, mAP at 0.5, delivered fraction, note
        ('Proposed, bit packed', 'Raspberry Pi 5', pi5_ms, fr_bit.mean() * ft, map_bit, del_bit, '10 % frame loss'),
        ('Proposed, byte aligned', 'Raspberry Pi 5', pi5_ms, fr_byte.mean() * ft, map_byte, del_byte, '10 % frame loss'),
        ('JPEG q50, no ARQ', 'Raspberry Pi 5', cx('jpeg_q50', 'pi5_auv_ms', 5.0), fr_jpeg.mean() * ft, map_jpeg, del_jpeg,
         '10 % frame loss, image lost if any frame is lost'),
        ('JPEG q50, selective repeat ARQ', 'Raspberry Pi 5', cx('jpeg_q50', 'pi5_auv_ms', 5.0),
         float(np.mean([arq_delivery_time(n, FER_OP) for n in fr_jpeg])), map_jpeg_free, 1.0, 'every image delivered'),
    ]
    if JETSON_LATENCY_MS is not None:
        schemes.append(('Proposed, bit packed', 'Jetson Orin NX 16GB', JETSON_LATENCY_MS, fr_bit.mean() * ft, map_bit,
                        del_bit, '10 % frame loss'))
    for label, (mdl, hw) in jscc_models.items():
        n_sym = (JSCC640_C if label == 'jscc640' else JSCC_C) * (hw // 8) ** 2
        spf = FRAME_PAYLOAD_BITS // JSCC_BITS_PER_SYMBOL
        schemes.append((f'{label} + detection at surface', 'Raspberry Pi 5', cx(label, 'pi5_auv_ms', np.nan),
                        math.ceil(n_sym / spf) * ft, jscc_map.get(label, np.nan), 1.0,
                        'SNR 10 dB AWGN, analog symbols degrade but are always delivered (Cell 14)'))
    if 'df_split' in globals():
        for _, r in df_split.iterrows():
            schemes.append((r['method'], 'Raspberry Pi 5', cx('split_l4_l6_l9', 'pi5_auv_ms', np.nan), r['airtime_s'],
                            r['map50'], (1 - FER_OP) ** r['frames'], 'features usable only if every frame arrives'))

    board_power = {'Raspberry Pi 5': PI5_POWER_W, 'Jetson Orin NX 16GB': JETSON_POWER_MODES_W[0]}
    board_price = {'Raspberry Pi 5': PI5_PRICE_USD, 'Jetson Orin NX 16GB': JETSON_PRICE_USD}
    rows = []
    for name, board, ms, air, mp, dfrac, note in schemes:
        if air is None or not np.isfinite(air) or air <= 0:
            print(f'Skipped {name}: no valid airtime (rerun the cell that produces it).')
            continue
        comp_s = ms / 1000.0 if (ms is not None and np.isfinite(ms)) else 0.0
        fps = min(1.0 / air, 1.0 / comp_s) if comp_s > 0 else 1.0 / air
        e_comp = board_power[board] * comp_s
        e_tx = TX_POWER_W * air
        e_img = e_comp + e_tx
        e_del = e_img / dfrac if dfrac > 1e-9 else np.inf
        rows.append(dict(scheme=name, board=board, compute_ms=ms, airtime_s=air, link_fps=1 / air,
                         compute_fps=1 / comp_s if comp_s > 0 else np.inf, achievable_fps=fps,
                         delivered_fraction=dfrac, delivered_fps=fps * dfrac, map50=mp,
                         compute_J_per_img=e_comp, transmit_J_per_img=e_tx, energy_J_per_sent_img=e_img,
                         energy_J_per_delivered_img=e_del,
                         battery_usd_per_1000_delivered=e_del / 3600 * 1000 * PRIMARY_BATTERY_USD_PER_WH,
                         delivered_images_per_mission_max=fps * dfrac * 3600 * MISSION_HOURS,
                         capex_usd=board_price[board] + CAMERA_PRICE_USD, note=note))
    df_cost = pd.DataFrame(rows)
    # Equal service mission, deliver as many images as reliable JPEG can in one mission
    n_common = float(df_cost.loc[df_cost.scheme == 'JPEG q50, selective repeat ARQ', 'delivered_images_per_mission_max'].iloc[0])
    feasible = df_cost.delivered_images_per_mission_max >= n_common * 0.999
    mission_wh = np.where(feasible, df_cost.energy_J_per_delivered_img * n_common / 3600 + MODEM_LISTEN_POWER_W * MISSION_HOURS,
                          np.nan)
    df_cost['equal_service_images_per_mission'] = n_common
    df_cost['equal_service_mission_Wh'] = mission_wh
    df_cost['equal_service_opex_3y_primary_usd'] = mission_wh * MISSIONS_PER_YEAR * YEARS * PRIMARY_BATTERY_USD_PER_WH
    df_cost['equal_service_opex_3y_grid_usd'] = mission_wh * MISSIONS_PER_YEAR * YEARS / 1000 * GRID_USD_PER_KWH
    df_cost['equal_service_tco_3y_usd'] = df_cost.capex_usd + df_cost.equal_service_opex_3y_primary_usd
    df_cost.to_csv(f'{OUT_DIR}/cost_energy_by_scheme.csv', index=False)
    print(df_cost.drop(columns=['note']).round(3).T.to_string())

    # Sensitivity, the conclusion must hold from a low board power up to the hard upper bound of the supply
    prop = df_cost[(df_cost.scheme == 'Proposed, bit packed') & (df_cost.board == 'Raspberry Pi 5')].iloc[0]
    jarq = df_cost[df_cost.scheme == 'JPEG q50, selective repeat ARQ'].iloc[0]
    sens = []
    for pw in [4.0, PI5_POWER_W, 15.0, PI5_POWER_MAX_W]:
        e_p = (pw * prop.compute_ms / 1000 + TX_POWER_W * prop.airtime_s) / prop.delivered_fraction
        e_j = pw * jarq.compute_ms / 1000 + TX_POWER_W * jarq.airtime_s
        sens.append(dict(board_power_W=pw, proposed_J_per_img=e_p, jpeg_arq_J_per_img=e_j, ratio=e_j / e_p))
    for rng_m, ptx in MODEM_TX_POWER_TABLE.items():
        e_p = (PI5_POWER_W * prop.compute_ms / 1000 + ptx * prop.airtime_s) / prop.delivered_fraction
        e_j = PI5_POWER_W * jarq.compute_ms / 1000 + ptx * jarq.airtime_s
        sens.append(dict(board_power_W=PI5_POWER_W, range_m=rng_m, tx_power_W=ptx, proposed_J_per_img=e_p,
                         jpeg_arq_J_per_img=e_j, ratio=e_j / e_p))
    df_cs = pd.DataFrame(sens)
    df_cs.to_csv(f'{OUT_DIR}/cost_energy_sensitivity.csv', index=False)
    print(df_cs.round(3).to_string(index=False))
    NUMBERS['energy_J_per_img_proposed'] = float(prop.energy_J_per_delivered_img)
    NUMBERS['energy_J_per_img_jpeg_arq'] = float(jarq.energy_J_per_delivered_img)
    NUMBERS['energy_ratio_jpeg_arq_vs_proposed'] = float(jarq.energy_J_per_delivered_img / prop.energy_J_per_delivered_img)
    NUMBERS['equal_service_tco_proposed_usd'] = float(prop.equal_service_tco_3y_usd)
    NUMBERS['equal_service_tco_jpeg_arq_usd'] = float(jarq.equal_service_tco_3y_usd)
    NUMBERS['energy_ratio_min_over_sensitivity'] = float(df_cs.ratio.min())
    NUMBERS['inference_energy_J'] = float(prop.compute_J_per_img)
    print(f"Energy per delivered image: proposed {prop.energy_J_per_delivered_img:.2f} J, reliable JPEG "
          f"{jarq.energy_J_per_delivered_img:.1f} J ({NUMBERS['energy_ratio_jpeg_arq_vs_proposed']:.0f}x). Ratio stays above "
          f"{NUMBERS['energy_ratio_min_over_sensitivity']:.0f}x over every board power and range setting.")
    print('A Raspberry Pi 5 runs the proposed scheme at the link-limited rate, so '
          'no NVIDIA GPU is required.')


In [ ]:
# Cell 24: 2D and 3D coverage
# Thorp absorption, ambient noise and practical spreading, calibrated to the modem datasheet power and range values.


def thorp_db_km(f_khz):
    f2 = f_khz ** 2
    return 0.11 * f2 / (1 + f2) + 44 * f2 / (4100 + f2) + 2.75e-4 * f2 + 0.003


def noise_psd_db(f_khz, s=SHIPPING_S, w=WIND_MPS):
    f = np.asarray(f_khz, float)
    nt = 17 - 30 * np.log10(f)
    ns = 40 + 20 * (s - 0.5) + 26 * np.log10(f) - 60 * np.log10(f + 0.03)
    nw = 50 + 7.5 * np.sqrt(w) + 20 * np.log10(f) - 40 * np.log10(f + 0.4)
    nth = -15 + 20 * np.log10(f)
    return 10 * np.log10(10 ** (nt / 10) + 10 ** (ns / 10) + 10 ** (nw / 10) + 10 ** (nth / 10))


_trapz = getattr(np, 'trapezoid', None) or getattr(np, 'trapz')
F_C_KHZ = (MODEM_F_LOW_KHZ + MODEM_F_HIGH_KHZ) / 2
BAND_HZ = (MODEM_F_HIGH_KHZ - MODEM_F_LOW_KHZ) * 1000


def noise_level_band_db(w=WIND_MPS):
    fg = np.linspace(MODEM_F_LOW_KHZ, MODEM_F_HIGH_KHZ, 400)
    return float(10 * np.log10(_trapz(10 ** (noise_psd_db(fg, w=w) / 10), fg * 1000)))


def esn0_db(d_m, p_w, l_impl=0.0, w=WIND_MPS):
    d = np.maximum(np.asarray(d_m, float), 1.0)
    sl = 170.8 + 10 * np.log10(TX_EFFICIENCY * np.asarray(p_w, float)) + DI_DB
    tl = SPREADING_K * 10 * np.log10(d) + d / 1000 * thorp_db_km(F_C_KHZ)
    rs = LINK_RATE_BPS / bits_per_symbol()
    return sl - tl - noise_level_band_db(w) - l_impl + 10 * np.log10(BAND_HZ / rs)


def esn0_required(fer=FER_REQ):
    lo, hi = -20.0, 60.0
    for _ in range(80):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if frame_error_rate(mid) > fer else (lo, mid)
    return hi


def tx_power_for(d_m):
    d = np.asarray(d_m, float)
    keys = sorted(MODEM_TX_POWER_TABLE)
    out = np.full(d.shape, MODEM_TX_POWER_MAX_W)
    for r in reversed(keys):
        out = np.where(d <= r, MODEM_TX_POWER_TABLE[r], out)
    return out


def max_range_m(p_w, l_impl, req, w=WIND_MPS):
    lo, hi = 1.0, 100000.0
    if esn0_db(lo, p_w, l_impl, w) < req:
        return 0.0
    for _ in range(80):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if esn0_db(mid, p_w, l_impl, w) >= req else (lo, mid)
    return lo


def max_covered_radius(covered_row, xs):
    # Largest horizontal distance up to which the row is covered without a gap, starting from the buoy
    if not covered_row[0]:
        return 0.0
    gap = np.where(~covered_row)[0]
    return float(xs[-1] if len(gap) == 0 else xs[gap[0] - 1])


if RUN_COVERAGE:
    REQ_ESN0 = esn0_required()
    margins = {r: float(esn0_db(r, p) - REQ_ESN0) for r, p in MODEM_TX_POWER_TABLE.items()}
    L_IMPL_DB = min(margins.values())
    spec('L_IMPL_DB', round(L_IMPL_DB, 2), 'dB', 'Propagation and coverage',
         'Calibrated so that every datasheet power-range pair meets FER_REQ (Cell 24)', 'measured')
    val = pd.DataFrame([dict(tx_power_W=p, datasheet_range_m=r, model_range_m=max_range_m(p, L_IMPL_DB, REQ_ESN0),
                             margin_before_calibration_dB=margins[r]) for r, p in MODEM_TX_POWER_TABLE.items()])
    val.to_csv(f'{OUT_DIR}/coverage_model_validation.csv', index=False)
    print(f'Required Es/N0 for FER {FER_REQ}: {REQ_ESN0:.2f} dB. Calibrated L_impl = {L_IMPL_DB:.1f} dB. '
          f'Band noise level {noise_level_band_db():.1f} dB re 1 uPa.')
    print(val.round(1).to_string(index=False))

    # Detection quality versus Es over N0 for every scheme, measured in Cell 14
    tb = pd.read_csv(f'{OUT_DIR}/ber_sweep_mean_std.csv')
    curves = {}
    for k in tb.key.unique():
        g = tb[tb.key == k].sort_values('snr')
        curves[k] = dict(snr=g.snr.values.astype(float), map=g.map50_mean.values, air=float(g.airtime_s.mean()),
                         frames=float(g.frames.mean()))
    compute_fps = {k: 1000.0 / PI5_LATENCY_MS for k in curves if k.startswith('proposed')}

    xs = np.arange(0, COVER_MAX_RANGE_M + 1, 25.0)
    zs = np.arange(0, COVER_MAX_DEPTH_M + 1, 10.0)
    X, Z = np.meshgrid(xs, zs)
    D = np.sqrt(X ** 2 + Z ** 2)
    P = tx_power_for(D)

    def scheme_maps(l_extra=0.0, w=WIND_MPS):
        es = esn0_db(D, P, L_IMPL_DB + l_extra, w)
        fer = 1 - (1 - np.vectorize(lambda e: ber_from_snr(float(e)))(es)) ** (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS)
        out = {}
        for k, cv in curves.items():
            m = np.interp(es, cv['snr'], cv['map'], left=cv['map'][0], right=cv['map'][-1])
            fps = np.full(es.shape, 1.0 / cv['air'])
            if k in compute_fps:
                fps = np.minimum(fps, compute_fps[k])
            out[k] = (m, fps)
        if 'jpeg' in curves:
            # Reliable JPEG with selective repeat ARQ, full quality but the delivery time grows with the frame loss
            nfj = int(round(curves['jpeg']['frames']))
            fer_lut = np.linspace(0, 0.999, 400)
            t_lut = np.array([arq_delivery_time(nfj, f) for f in fer_lut])
            t_arq = np.interp(np.clip(fer, 0, 0.999), fer_lut, t_lut)
            out['jpeg_arq'] = (np.full(es.shape, curves['jpeg']['map'][-1]), 1.0 / t_arq)
        return es, out

    ES, MAPS = scheme_maps()
    summ = []
    depths = [0, 50, 100, 200, 500, 1000]
    for k, (m, fps) in MAPS.items():
        for ft_ in FPS_TARGETS:
            cov = (m >= MAP_TARGET) & (fps >= ft_) & (D <= MODEM_MAX_RANGE_M)
            radii = np.array([max_covered_radius(cov[i], xs) for i in range(len(zs))])
            for dz in depths:
                i = int(np.argmin(np.abs(zs - dz)))
                summ.append(dict(scheme=k, fps_target=ft_, depth_m=dz, radius_m=radii[i],
                                 area_km2=np.pi * (radii[i] / 1000) ** 2))
            dzk = (zs[1] - zs[0]) / 1000
            vol200 = float(np.sum(np.pi * (radii[zs <= 200] / 1000) ** 2) * dzk)
            vol_all = float(np.sum(np.pi * (radii / 1000) ** 2) * dzk)
            summ.append(dict(scheme=k, fps_target=ft_, depth_m='volume 0-200 m', radius_m=np.nan, area_km2=np.nan,
                             volume_km3=vol200))
            summ.append(dict(scheme=k, fps_target=ft_, depth_m=f'volume 0-{int(zs[-1])} m', radius_m=np.nan,
                             area_km2=np.nan, volume_km3=vol_all))
    df_cov = pd.DataFrame(summ)
    df_cov.to_csv(f'{OUT_DIR}/coverage_summary.csv', index=False)
    print(df_cov.round(3).to_string(index=False))

    # 2D maps of the delivered frame rate where mAP meets MAP_TARGET (grey elsewhere)
    from matplotlib.colors import LogNorm
    keys = list(MAPS.keys())
    ncol = min(3, len(keys)); nrow = int(math.ceil(len(keys) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.8 * nrow), squeeze=False, constrained_layout=True)
    for ax, k in zip(axs.flat, keys):
        m, fps = MAPS[k]
        shown = np.where((m >= MAP_TARGET) & (D <= MODEM_MAX_RANGE_M), fps, np.nan)
        ax.set_facecolor('0.85')
        im = ax.imshow(shown, origin='upper', aspect='auto', cmap='viridis', norm=LogNorm(vmin=0.01, vmax=20),
                       extent=[xs[0] / 1000, xs[-1] / 1000, zs[-1], zs[0]])
        ax.set_title(k, fontsize=9)
        ax.set_xlabel('Horizontal range (km)'); ax.set_ylabel('AUV depth (m)')
    for ax in list(axs.flat)[len(keys):]:
        ax.axis('off')
    fig.colorbar(im, ax=axs, shrink=0.8, label=f'Frames per second delivered with mAP@0.5 >= {MAP_TARGET}')
    plt.savefig(f'{OUT_DIR}/fig_coverage_maps.png', dpi=300, bbox_inches='tight'); plt.show()

    # mAP versus horizontal range at 50 m depth
    i50 = int(np.argmin(np.abs(zs - 50)))
    plt.figure(figsize=(6, 4))
    for k, (m, fps) in MAPS.items():
        ok = (fps[i50] >= FPS_TARGETS[0]) & (D[i50] <= MODEM_MAX_RANGE_M)   # nothing beyond the datasheet range
        plt.plot(xs / 1000, np.where(ok, m[i50], np.nan), label=k)
    plt.axvline(MODEM_MAX_RANGE_M / 1000, color='r', ls=':', lw=0.8)
    plt.axhline(MAP_TARGET, color='k', ls='--', lw=0.8)
    plt.xlim(0, MODEM_MAX_RANGE_M / 1000 * 1.05)
    plt.xlabel('Horizontal range (km), AUV at 50 m depth (red: datasheet maximum range)')
    plt.ylabel(f'mAP@0.5 (where >= {FPS_TARGETS[0]} fps)')
    plt.grid(alpha=0.3); plt.legend(fontsize=7); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/fig_map_vs_range.png', dpi=300); plt.show()

    # AUVs per surface buoy with TDMA, worst case and delay compensated guard times
    airs = {k: cv['air'] for k, cv in curves.items()}
    if 'jpeg' in curves:
        airs['jpeg_arq'] = arq_delivery_time(int(round(curves['jpeg']['frames'])), FER_REQ)
    cap = []
    for gname, guard in [('worst_case', OPERATING_RANGE_M / SOUND_SPEED_MPS), ('delay_compensated', TDMA_GUARD_COMP_S)]:
        for k, air in airs.items():
            for ft_ in FPS_TARGETS:
                cap.append(dict(scheme=k, guard=gname, guard_s=guard, fps_target=ft_, slot_s=air + guard,
                                auvs_per_buoy=int(np.floor(1.0 / (ft_ * (air + guard))))))
    df_cap = pd.DataFrame(cap)
    df_cap.to_csv(f'{OUT_DIR}/multi_auv_capacity.csv', index=False)
    print(df_cap.round(3).to_string(index=False))

    # Sensitivity of the coverage radius at 100 m depth to extra loss and to wind noise
    sens = []
    i100 = int(np.argmin(np.abs(zs - 100)))
    for l_extra in [-3.0, 0.0, 3.0, 6.0]:
        for w in [0.0, WIND_MPS, 10.0, 15.0]:
            _, mp = scheme_maps(l_extra, w)
            for k, (m, fps) in mp.items():
                cov = (m >= MAP_TARGET) & (fps >= FPS_TARGETS[0]) & (D <= MODEM_MAX_RANGE_M)
                sens.append(dict(extra_loss_dB=l_extra, wind_mps=w, scheme=k,
                                 radius_m_at_100m=max_covered_radius(cov[i100], xs)))
    df_csens = pd.DataFrame(sens)
    df_csens.to_csv(f'{OUT_DIR}/coverage_sensitivity.csv', index=False)
    print(df_csens.pivot_table(index=['extra_loss_dB', 'wind_mps'], columns='scheme', values='radius_m_at_100m').to_string())
    for k in MAPS:
        for ft_ in FPS_TARGETS:
            r = df_cov[(df_cov.scheme == k) & (df_cov.fps_target == ft_) & (df_cov.depth_m == 100)]
            if len(r):
                NUMBERS[f'coverage_radius_m_100m_{ft_}fps_{k}'] = float(r.radius_m.iloc[0])
    NUMBERS['L_impl_dB'] = L_IMPL_DB


In [ ]:
# Cell 25: Software in the loop link emulation
# Real bit packed frames with header and CRC-32 are sent over UDP with bit errors at the datasheet frame timing.
SIL_SNRS_DB = spec('SIL_SNRS_DB', [8, 12], 'dB Es/N0', 'Evaluation protocol',
                   'Inside and above the frame error transition of Cell 14', 'design choice')
SIL_EDGE_MS = PI5_LATENCY_MS
SIL_MAX_IMAGES = None   # None equals all validation images about 2 to 3 minutes of real time per SNR
FRAME_BYTES = (FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS) // 8


def build_frames(img_id, b, s, c, w, h, nc):
    rec_bits = 40 + class_bits(nc) + 8
    per = FRAME_PAYLOAD_BITS // rec_bits
    order = np.argsort(-np.asarray(s), kind='stable')   # most confident detections first
    n = len(b); nf = max(1, int(math.ceil(n / per)))
    if nf > 16:
        nf = 16; order = order[:16 * per]               # 4 bit frame index, never reached on URPC2020
    frames = []
    for k in range(nf):
        idx = order[k * per:(k + 1) * per]
        data, _ = pack_bitpacked(b[idx], s[idx], c[idx], w, h, nc)
        payload = data.ljust(FRAME_PAYLOAD_BITS // 8, b'\x00')
        hdr = ((img_id & 0xFFFFF) << 12) | ((k & 0xF) << 8) | (len(idx) & 0xFF)
        body = struct.pack('>I', hdr) + payload
        frames.append(body + struct.pack('>I', zlib.crc32(body) & 0xFFFFFFFF))
    return frames


def corrupt_bits(frame, ber, rng):
    bits = np.unpackbits(np.frombuffer(frame, np.uint8))
    flips = (rng.rand(bits.size) < ber).astype(np.uint8)
    return np.packbits(np.bitwise_xor(bits, flips)).tobytes()


def parse_frame(fr, nc, wh):
    body, crc = fr[:-4], struct.unpack('>I', fr[-4:])[0]
    if (zlib.crc32(body) & 0xFFFFFFFF) != crc:
        return None
    hdr = struct.unpack('>I', body[:4])[0]
    img_id, k, n = hdr >> 12, (hdr >> 8) & 0xF, hdr & 0xFF
    if img_id not in wh:
        return None
    w, h = wh[img_id]
    nbytes = int(math.ceil(n * (40 + class_bits(nc) + 8) / 8))
    bb, ss, cc = unpack_bitpacked(body[4:4 + nbytes], n, nc, w, h)
    return img_id, k, bb, ss, cc


def run_sil(cache, es_n0_db, seed=42, nc=NC, edge_ms=SIL_EDGE_MS, range_m=OPERATING_RANGE_M):
    ber = ber_from_snr(es_n0_db); ft = frame_time_s(); prop = range_m / SOUND_SPEED_MPS
    rx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); rx.bind(('127.0.0.1', 0)); rx.settimeout(0.5)
    rx.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 1 << 22)
    port = rx.getsockname()[1]
    tx = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    txq = queue.Queue(maxsize=1)
    wh = {it['idx']: (it['w'], it['h']) for it in cache}
    cap_t, last_t, recv = {}, {}, {}
    st = dict(sent=0, crc_fail=0)
    tx_done = threading.Event()

    def auv():
        for it in cache:
            t0 = time.perf_counter(); cap_t[it['idx']] = t0
            frames = build_frames(it['idx'], it['det_b'], it['det_s'], it['det_c'], it['w'], it['h'], nc)
            rest = edge_ms / 1000.0 - (time.perf_counter() - t0)
            if rest > 0:
                time.sleep(rest)
            txq.put(frames)
        txq.put(None)

    def modem():
        rng = np.random.RandomState([int(seed), 4242])
        t_next = time.perf_counter()
        while True:
            frames = txq.get()
            if frames is None:
                break
            for fr in frames:
                t_next = max(t_next, time.perf_counter())
                tx.sendto(corrupt_bits(fr, ber, rng), ('127.0.0.1', port)); st['sent'] += 1
                t_next += ft
                d = t_next - time.perf_counter()
                if d > 0:
                    time.sleep(d)
        tx_done.set()

    def surface():
        while True:
            try:
                data, _ = rx.recvfrom(4096)
            except socket.timeout:
                if tx_done.is_set():
                    break
                continue
            t_arr = time.perf_counter() + ft + prop      # frame fully received after its airtime and propagation
            out = parse_frame(data, nc, wh)
            if out is None:
                st['crc_fail'] += 1; continue
            img_id, k, bb, ss, cc = out
            recv.setdefault(img_id, []).append((bb, ss, cc)); last_t[img_id] = t_arr

    th = [threading.Thread(target=f, daemon=True) for f in (surface, modem, auv)]
    t_start = time.perf_counter()
    for t_ in th:
        t_.start()
    for t_ in th[::-1]:
        t_.join()
    rx.close(); tx.close()
    preds, lat = [], []
    for it in cache:
        parts = recv.get(it['idx'], [])
        if parts:
            preds.append(make_pred_dict(np.concatenate([p[0] for p in parts]).reshape(-1, 4),
                                        np.concatenate([p[1] for p in parts]), np.concatenate([p[2] for p in parts])))
            lat.append(last_t[it['idx']] - cap_t[it['idx']])
        else:
            preds.append(make_pred_dict(np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
    m = compute_map_full(preds, gts_of(cache))
    t_end = max(last_t.values()) if last_t else time.perf_counter()
    return dict(es_n0_db=es_n0_db, images=len(cache), frames_sent=st['sent'],
                frame_error_rate_measured=st['crc_fail'] / max(st['sent'], 1),
                frame_error_rate_theory=frame_error_rate(es_n0_db), **m,
                delivered_fps=len(cache) / (t_end - t_start), e2e_delay_median_s=float(np.median(lat)) if lat else np.nan,
                e2e_delay_p95_s=float(np.percentile(lat, 95)) if lat else np.nan, edge_latency_ms=edge_ms,
                one_way_propagation_s=prop), lat


if RUN_SIL:
    sil_cache = cache_urpc if SIL_MAX_IMAGES is None else cache_urpc[:SIL_MAX_IMAGES]
    rows, lats = [], {}
    for snr in SIL_SNRS_DB:
        r, lat = run_sil(sil_cache, snr)
        sim = run_condition(sil_cache, 'proposed', {'type': 'ber', 'snr_db': snr}, 42, fmt='fixed_bitpacked')
        r['map50_simulation_same_snr'] = sim['map50']
        rows.append(r); lats[snr] = lat
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
    df_sil = pd.DataFrame(rows)
    df_sil.to_csv(f'{OUT_DIR}/sil_link_emulation.csv', index=False)
    plt.figure(figsize=(6, 4))
    for snr, lat in lats.items():
        plt.hist(lat, bins=40, alpha=0.6, label=f'Es/N0 {snr} dB')
    plt.xlabel('End-to-end delay per image (s): capture to last frame at the surface'); plt.ylabel('Images')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/fig_sil_delay.png', dpi=300); plt.show()
    NUMBERS['sil_delivered_fps'] = float(df_sil.delivered_fps.iloc[-1])
    NUMBERS['sil_e2e_delay_median_s'] = float(df_sil.e2e_delay_median_s.iloc[-1])


In [ ]:
# Cell 26: Detector training seeds (session B)
if RUN_TRAIN_SEEDS:
    rows = []
    for ts in [GLOBAL_SEED] + TRAIN_SEEDS:
        mdl, _ = (detector, DET_WEIGHTS) if ts == GLOBAL_SEED else \
            train_or_load(f'train_n_seed{ts}', 'yolov8n.pt', URPC_YAML, ts)
        c_ = cache_urpc if ts == GLOBAL_SEED else \
            build_det_cache(mdl, VAL_FILES, VAL_LBL, f'{OUT_DIR}/cache_urpc_n640_seed{ts}.pkl')
        rows.append(dict(run_condition(c_, 'oracle', {'type': 'none'}, 42), train_seed=ts, cond='oracle'))
        for sd in LOSS_SEEDS:
            rows.append(dict(run_condition(c_, 'proposed', {'type': 'iid', 'rho': 0.10}, sd, legacy=True),
                             train_seed=ts, cond='iid_0.10_legacy'))
            rows.append(dict(run_condition(c_, 'proposed', {'type': 'ber', 'snr_db': 8}, sd, fmt='fixed_bitpacked'),
                             train_seed=ts, cond='ber_8dB_bitpacked'))
    df_ts = pd.DataFrame(rows)
    t = agg(df_ts, ['method', 'cond'])
    t.to_csv(f'{OUT_DIR}/training_seeds_mean_std.csv', index=False)
    print(t.round(4).to_string(index=False))


In [ ]:
# Cell 27: RUOD cross dataset evaluation (session C)
RUOD_SHARED = {'echinus': ['echinus', 'sea urchin', 'urchin'], 'holothurian': ['holothurian', 'sea cucumber'],
               'scallop': ['scallop'], 'starfish': ['starfish', 'sea star']}


def find_ruod_jsons():
    # Uses RUOD_ANN only; the Environment folders hold per condition subsets
    js = [p for p in glob.glob(f'{INPUT_DIR}/**/*.json', recursive=True)
          if 'ruod' in p.lower() and 'environ' not in p.lower()]
    main = [p for p in js if 'ruod_ann' in p.lower()] or js
    tr = sorted(p for p in main if 'train' in os.path.basename(p).lower())
    te = sorted(p for p in main if any(t in os.path.basename(p).lower() for t in ('test', 'val')))
    if not tr or not te:
        raise FileNotFoundError('RUOD train and test annotation json files not found under /kaggle/input. '
                                f'JSON files seen: {main[:10]}')
    print('RUOD annotations:', tr[0], '|', te[0])
    return tr[0], te[0]


def coco_to_yolo(json_path, img_index, out_root, split):
    d = json.load(open(json_path))
    cats = sorted(d['categories'], key=lambda c: c['id'])
    cmap = {c['id']: i for i, c in enumerate(cats)}
    names_ = [c['name'] for c in cats]
    anns = {}
    for a in d['annotations']:
        anns.setdefault(a['image_id'], []).append(a)
    idir = f'{out_root}/{split}/images'; ldir = f'{out_root}/{split}/labels'
    os.makedirs(idir, exist_ok=True); os.makedirs(ldir, exist_ok=True)
    for im in d['images']:
        src_ = img_index.get(os.path.basename(im['file_name']))
        if src_ is None:
            continue
        dst = f'{idir}/{os.path.basename(src_)}'
        if not os.path.exists(dst):
            os.symlink(src_, dst)
        W, H = im['width'], im['height']
        with open(f'{ldir}/{os.path.splitext(os.path.basename(src_))[0]}.txt', 'w') as f:
            for a in anns.get(im['id'], []):
                x, y, w, h = a['bbox']
                if w <= 0 or h <= 0:
                    continue
                f.write(f"{cmap[a['category_id']]} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}\n")
    return names_


if RUN_RUOD:
    tr_json, te_json = find_ruod_jsons()
    img_index = {}
    for root, _, files in os.walk(INPUT_DIR):
        rl = root.lower()
        if 'ruod' in rl and 'environ' not in rl:      # use the RUOD_pic train and test folders only
            for fn in files:
                if fn.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_index.setdefault(fn, os.path.join(root, fn))
    print('RUOD images indexed:', len(img_index))
    RUOD_DIR = f'{WORK_DIR}/ruod_yolo'
    ruod_names = coco_to_yolo(tr_json, img_index, RUOD_DIR, 'train')
    coco_to_yolo(te_json, img_index, RUOD_DIR, 'val')
    RUOD_YAML = f'{RUOD_DIR}/data.yaml'
    yaml.safe_dump({'path': RUOD_DIR, 'train': f'{RUOD_DIR}/train/images', 'val': f'{RUOD_DIR}/val/images',
                    'nc': len(ruod_names), 'names': ruod_names}, open(RUOD_YAML, 'w'))
    RUOD_VAL = sorted(glob.glob(f'{RUOD_DIR}/val/images/*'))
    NC_RUOD = len(ruod_names)
    print('RUOD classes:', ruod_names, '| test images:', len(RUOD_VAL))

    # Zero shot, URPC detector on RUOD, restricted to the four shared classes domain shift, Section VII D
    c_zs = build_det_cache(detector, RUOD_VAL, f'{RUOD_DIR}/val/labels', f'{OUT_DIR}/cache_ruod_zeroshot.pkl',
                           jpeg_qualities=())
    to_ruod = {}
    for ui, un in enumerate(CLASS_NAMES):
        for ri, rn in enumerate(ruod_names):
            if rn.lower() in RUOD_SHARED.get(un.lower(), [un.lower()]):
                to_ruod[ui] = ri
    shared = set(to_ruod.values())
    preds, gts = [], []
    for it in c_zs:
        keep = np.array([c in to_ruod for c in it['det_c']], bool)
        preds.append(make_pred_dict(it['det_b'][keep], it['det_s'][keep],
                                    np.array([to_ruod[c] for c in it['det_c'][keep]], int)))
        g = np.array([c in shared for c in it['gt_c']], bool)
        gts.append(make_pred_dict(it['gt_b'][g], np.ones(int(g.sum())), it['gt_c'][g]))
    NUMBERS['ruod_zero_shot_map50_shared4'] = compute_map_full(preds, gts)['map50']
    print('Zero shot mAP@0.5 on shared classes:', round(NUMBERS['ruod_zero_shot_map50_shared4'], 4))

    # Trained on RUOD, then the same channel experiments
    det_ruod, _ = train_or_load('train_n_ruod', 'yolov8n.pt', RUOD_YAML, GLOBAL_SEED, epochs=RUOD_EPOCHS)
    c_r = build_det_cache(det_ruod, RUOD_VAL, f'{RUOD_DIR}/val/labels', f'{OUT_DIR}/cache_ruod_n640.pkl',
                          jpeg_qualities=(50,))
    rows = [dict(run_condition(c_r, 'oracle', {'type': 'none'}, 42, nc=NC_RUOD), cond='oracle')]
    for sd in LOSS_SEEDS:
        for cond, lm in [('iid_0.10', {'type': 'iid', 'rho': 0.10}), ('ber_8dB', {'type': 'ber', 'snr_db': 8})]:
            for fmt in ['fixed_bytealigned', 'fixed_bitpacked']:
                rows.append(dict(run_condition(c_r, 'proposed', lm, sd, fmt=fmt, nc=NC_RUOD), cond=cond))
            rows.append(dict(run_condition(c_r, 'jpeg', lm, sd, nc=NC_RUOD), cond=cond))
    t = agg(pd.DataFrame(rows), ['method', 'fmt', 'cond'])
    t.to_csv(f'{OUT_DIR}/ruod_results_mean_std.csv', index=False)
    print(t.round(4).to_string(index=False))


In [ ]:
# Cell 28: Parameter table, platform figure and operating procedure
from matplotlib.patches import FancyBboxPatch


def tex_escape(s):
    s = str(s)
    for a, b in [('\\', r'\textbackslash{}'), ('&', r'\&'), ('%', r'\%'), ('$', r'\$'), ('#', r'\#'),
                 ('_', r'\_'), ('{', r'\{'), ('}', r'\}'), ('~', r'\textasciitilde{}'), ('^', r'\^{}')]:
        s = s.replace(a, b)
    return s


df_spec = pd.DataFrame(SPEC_ROWS)
df_spec['value'] = df_spec.value.map(lambda v: json.dumps(v) if isinstance(v, (dict, list, tuple)) else v)
df_spec.to_csv(f'{OUT_DIR}/parameter_table.csv', index=False)
with open(f'{OUT_DIR}/parameter_table.tex', 'w') as f:
    f.write('\\begin{table*}[t]\n\\centering\n\\caption{System parameters, settings and sources}\n')
    f.write('\\label{tab:params}\n\\footnotesize\n\\begin{tabular}{p{3.3cm}p{2.6cm}p{1.6cm}p{6.6cm}p{2.0cm}}\n\\hline\n')
    f.write('Parameter & Value & Unit & Source / setting support & Kind \\\\\n\\hline\n')
    for grp, g in df_spec.groupby('group', sort=False):
        f.write(f'\\multicolumn{{5}}{{l}}{{\\textit{{{tex_escape(grp)}}}}} \\\\\n')
        for _, r in g.iterrows():
            f.write(' & '.join(tex_escape(x) for x in (r.key, r.value, r.unit, r.source, r.kind)) + ' \\\\\n')
    f.write('\\hline\n\\end{tabular}\n\\end{table*}\n')
print(df_spec.groupby('kind').size().to_string())
print('Parameters of kind assumption (each has a sensitivity analysis):',
      ', '.join(df_spec[df_spec.kind == 'assumption'].key))

# Platform figure
fig, ax = plt.subplots(figsize=(14, 4.6)); ax.axis('off'); ax.set_xlim(0, 14); ax.set_ylim(0, 4.6)
blocks = [
    (0.1, 'Camera', CAMERA_NAME.replace(' Camera ', '\nCamera '), 'I'),
    (2.1, 'Edge board', f'Raspberry Pi 5\nYOLOv8-n (NCNN)\n{PI5_LATENCY_MS:.0f} ms/img', 'D(I) = S'),
    (4.1, 'Descriptor encoder', f'Alg. 1, bit packed\n{FORMATS["fixed_bitpacked"](NC)} bit/detection', 'enc(S)'),
    (6.1, 'Framer', f'{(FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS) // 8}-byte frames\nheader + CRC-32', 'x_k'),
    (8.1, 'Acoustic modem', f'{MODEM_NAME}\n{MODEM_F_LOW_KHZ:.0f}-{MODEM_F_HIGH_KHZ:.0f} kHz, {LINK_RATE_BPS / 1000:.0f} kbit/s\n'
                            f'{TX_POWER_W} W at {OPERATING_RANGE_M} m', 'acoustic signal'),
    (10.1, 'Channel', 'Thorp absorption\nambient noise, BER\nGilbert-Elliott bursts', 'y_k'),
    (12.1, 'Surface station', 'modem, CRC check\ndecoder', 'S_hat'),
]
for x0, title, body, sig in blocks:
    ax.add_patch(FancyBboxPatch((x0, 1.4), 1.7, 2.0, boxstyle='round,pad=0.05', fc='#eef3fb', ec='#2b5797'))
    ax.text(x0 + 0.85, 3.15, title, ha='center', va='center', fontsize=9, weight='bold')
    ax.text(x0 + 0.85, 2.2, body, ha='center', va='center', fontsize=7.2)
    if x0 < 12:
        ax.annotate('', xy=(x0 + 2.0, 2.4), xytext=(x0 + 1.8, 2.4), arrowprops=dict(arrowstyle='->', lw=1.4))
        ax.text(x0 + 1.9, 3.55, sig, ha='center', fontsize=8, style='italic')
ax.text(0.1, 0.9, 'AUV node (left of the channel): all values from the datasheets and benchmark listed in Table '
        '"System parameters". Surface station decodes the detection list S_hat.', fontsize=8)
ax.text(0.1, 0.45, f'Measured on this pipeline: T4 latency {NUMBERS.get("t4_latency_ms_median", float("nan")):.2f} ms; '
        f'emulated end-to-end delay {NUMBERS.get("sil_e2e_delay_median_s", float("nan")):.2f} s; '
        f'delivered rate {NUMBERS.get("sil_delivered_fps", float("nan")):.1f} fps (Cell 25).', fontsize=8)
plt.savefig(f'{OUT_DIR}/fig_platform.png', dpi=300, bbox_inches='tight'); plt.show()

PROCEDURE = f"""# Operating procedure

1. Data. Download URPC2020 (4 classes; {len(VAL_FILES)} validation images) and RUOD (github.com/dlut-dimt/RUOD).
   Upload both as private Kaggle datasets. Validation images are processed in alphabetical order.
2. Detector. Fine-tune YOLOv8-n for {YOLO_EPOCHS} epochs at {IMGSZ} px, batch 16, AdamW (lr 0.00125, selected automatically by Ultralytics), seed 42,
   deterministic=True
   (Cell 5). Seeds 43 and 44 are trained in session B.
3. Detection cache. Run the detector once per image with confidence {CONF_THR} and NMS IoU {NMS_IOU_THR}, after
   20 warm-up images; store boxes, scores, classes, latency and JPEG sizes (Cell 9).
4. Descriptor encoding. Quantize to Q_POS = {Q_POS}, Q_CONF = {Q_CONF}; bit-pack
   ({FORMATS['fixed_bitpacked'](NC)} bit/detection) and group into {FRAME_PAYLOAD_BITS // 8}-byte payloads (Cell 7).
5. Channel. Frame error rate from Es/N0 ({MODULATION.upper()}, {FRAME_PAYLOAD_BITS + FRAME_OVERHEAD_BITS}-bit frames);
   iid erasure and Gilbert-Elliott bursts (Cell 8). Seeds {LOSS_SEEDS}; report mean and standard deviation.
6. Baselines. JPEG q25/50/75 (with and without ARQ), JSCC autoencoder, DeepJSCC, split computing (Cells 11-21).
7. Link emulation. Send the real frames through the software-in-the-loop emulator at the datasheet frame timing,
   with bit errors and CRC-32 checking (Cell 25).
8. Physical layer and coverage. Thorp absorption, ambient noise and practical spreading, calibrated to the
   {MODEM_NAME} power-range datasheet values; coverage maps and multi-AUV capacity (Cell 24).
9. Cost and energy. Energy per delivered image and 3-year TCO per scheme from datasheet values (Cell 23).
10. Release. run_config.json, requirements_frozen.txt, weights and all CSV files are archived in the release zip (Cell 30).
"""
with open(f'{OUT_DIR}/operating_procedure.md', 'w') as f:
    f.write(PROCEDURE)
print(PROCEDURE)


In [ ]:
# Cell 29: Optional edge export and benchmark script
EDGE_DIR = f'{OUT_DIR}/edge'
os.makedirs(EDGE_DIR, exist_ok=True)
try:
    onnx_path = YOLO(DET_WEIGHTS).export(format='onnx', imgsz=IMGSZ, opset=12)
    shutil.copy(onnx_path, EDGE_DIR)
except Exception as e:
    print('ONNX export skipped:', e)
for p in VAL_FILES[:50]:
    os.makedirs(f'{EDGE_DIR}/sample_images', exist_ok=True)
    shutil.copy(p, f'{EDGE_DIR}/sample_images/')
shutil.copy(DET_WEIGHTS, f'{EDGE_DIR}/best.pt')

EDGE_SCRIPT = r'''
# On-device latency benchmark. Copy the edge folder to the device and run:
#   Raspberry Pi 5:  pip install ultralytics ncnn
#                    python edge_benchmark.py --weights best.pt --format ncnn
#   Jetson:          pip install ultralytics   (JetPack with TensorRT)
#                    python edge_benchmark.py --weights best.pt --format engine --tegrastats
# Power: on Jetson the script logs tegrastats. On the Pi read the inline USB meter during the run.
import argparse, glob, json, os, subprocess, time
import numpy as np
from ultralytics import YOLO

ap = argparse.ArgumentParser()
ap.add_argument("--weights", default="best.pt")
ap.add_argument("--format", default="ncnn", choices=["pt", "onnx", "ncnn", "engine"])
ap.add_argument("--imgsz", type=int, default=640)
ap.add_argument("--frames", type=int, default=300)
ap.add_argument("--tegrastats", action="store_true")
a = ap.parse_args()

model_path = a.weights
if a.format != "pt":
    kw = {"half": True} if a.format == "engine" else {}
    model_path = YOLO(a.weights).export(format=a.format, imgsz=a.imgsz, **kw)
model = YOLO(model_path, task="detect")
imgs = sorted(glob.glob("sample_images/*"))
proc = None
if a.tegrastats:
    log = open("tegrastats.log", "w")
    proc = subprocess.Popen(["tegrastats", "--interval", "100"], stdout=log)
for p in imgs[:20]:
    model(p, imgsz=a.imgsz, conf=0.10, iou=0.45, verbose=False)
lat = []
for k in range(a.frames):
    t0 = time.perf_counter()
    model(imgs[k % len(imgs)], imgsz=a.imgsz, conf=0.10, iou=0.45, verbose=False)
    lat.append((time.perf_counter() - t0) * 1000)
if proc:
    proc.terminate()
res = {"format": a.format, "imgsz": a.imgsz, "frames": a.frames,
       "latency_ms_mean": float(np.mean(lat)), "latency_ms_std": float(np.std(lat)),
       "latency_ms_p95": float(np.percentile(lat, 95)), "fps": float(1000 / np.mean(lat))}
if a.tegrastats and os.path.exists("tegrastats.log"):
    import re
    mw = [int(m.group(1)) for m in re.finditer(r"VDD_IN (\d+)mW", open("tegrastats.log").read())]
    if mw:
        res["power_w_mean"] = float(np.mean(mw) / 1000)
print(json.dumps(res, indent=2))
json.dump(res, open("edge_result.json", "w"), indent=2)
'''
with open(f'{EDGE_DIR}/edge_benchmark.py', 'w') as f:
    f.write(EDGE_SCRIPT)
print('Edge package written to', EDGE_DIR)


In [ ]:
# Cell 30: Release package
import subprocess
config = {k: v for k, v in globals().items()
          if k.isupper() and not k.startswith('_') and isinstance(v, (int, float, str, list, tuple, bool, dict))
          and k not in ('NUMBERS', 'MAPS', 'MAP', 'COMPLEX', 'FORMATS', 'PROCEDURE', 'SPEC_ROWS', 'VAL_FILES',
                        'RUOD_VAL', 'EDGE_SCRIPT')
          and not (isinstance(v, (list, tuple, dict)) and len(v) > 100)}
config['environment'] = {'python': platform.python_version(), 'torch': torch.__version__,
                         'ultralytics': __import__('ultralytics').__version__,
                         'gpu': torch.cuda.get_device_name(0) if device == 'cuda' else 'cpu'}
json.dump(config, open(f'{OUT_DIR}/run_config.json', 'w'), indent=2, default=str)
json.dump({k: (float(v) if isinstance(v, (np.floating, float, int)) else v) for k, v in NUMBERS.items()},
          open(f'{OUT_DIR}/results_summary.json', 'w'), indent=2, default=str)
subprocess.run(f'pip freeze > {OUT_DIR}/requirements_frozen.txt', shell=True)
shutil.copy(DET_WEIGHTS, f'{OUT_DIR}/yolov8n_urpc_best.pt')
# Detection caches (*.pkl) are left out; Cell 9 rebuilds them.
REL = f'{WORK_DIR}/results_session_{_S}'
shutil.rmtree(REL, ignore_errors=True)
shutil.copytree(OUT_DIR, REL, ignore=shutil.ignore_patterns('*.pkl', '*.ckpt', '*.tmp'))
for extra in ['results.png', 'results.csv', 'confusion_matrix.png', 'args.yaml']:
    p = os.path.join(os.path.dirname(os.path.dirname(DET_WEIGHTS)), extra)
    if os.path.exists(p):
        shutil.copy(p, f'{REL}/detector_{extra}')
shutil.make_archive(REL, 'zip', REL)
n_png = len(glob.glob(f'{REL}/**/*.png', recursive=True)); n_csv = len(glob.glob(f'{REL}/**/*.csv', recursive=True))
print(f'Release archive: {REL}.zip  ({n_png} figures, {n_csv} tables, '
      f'{os.path.getsize(REL + ".zip") / 1e6:.1f} MB)')
print(json.dumps({k: round(float(v), 4) if isinstance(v, (float, np.floating)) else v
                  for k, v in NUMBERS.items()}, indent=1, default=str))
